# GradientBoost 전용 실습 — Mercari 가격 예측 데이터 기준

이 노트북은 `10 캐글 mercari price_정제후.ipynb`가 다루는 **동일한 데이터(`mercari_train.tsv`)**를 기준으로 하되, **GradientBoost 계열 모델(GradientBoostingRegressor / GradientBoostingClassifier) 두 가지만** 다루는 신규 노트북이다. Ridge·LightGBM 등 기존에 이미 실행했던 모델은 이 노트북에 포함하지 않는다.

| 항목 | 내용 |
|---|---|
| 데이터 | `mercari_train.tsv`(1,482,535행) 전체 — 무작위 표본 미사용, 전수 데이터 사용(2026-08-26 변경) |
| Part A | `GradientBoostingRegressor` — **가격(price, 연속값)** 예측 |
| Part B | `GradientBoostingClassifier` — **배송비 부담 주체(shipping, 0/1 이진)** 분류 |
| Part C | 두 모듈의 차이(같은 알고리즘, 다른 목적함수) 비교 |
| Part D | 내부테스트(자체 검증) — 결과값이 정상 범위인지 코드로 직접 검증 |

> 두 모델(Part A/B)은 **동일한 피처 엔지니어링 파이프라인**을 공유한다 — 단, `shipping`은 Part B의 분류 타깃이므로 두 모델 모두 피처 목록에서 제외했다(§2-4에서 이유 설명, 정답 누출 방지 + 두 모델 간 공정한 비교를 위해 피처셋을 동일하게 유지).

## 1. 데이터 로드(전체 데이터 사용, 표본 추출 없음)

In [2]:
# 기술적 의미: 표 형태 데이터를 다루는 pandas 라이브러리를 pd라는 별칭으로 불러온다.
# 업무적 의미: 이후 모든 데이터 적재·가공·분석은 pandas의 DataFrame을 기본 단위로 수행한다 — 이 프로젝트 데이터 작업의 공통 전제.
import pandas as pd

# 기술적 의미: 탭 문자(\t)로 구분된 mercari_train.tsv 파일을 읽어 DataFrame으로 만든다. 경로는 이 노트북 파일(ipynb/src/) 기준 상대경로다.
# 업무적 의미: 실제 캐글 Mercari 중고거래 플랫폼의 판매 이력(148만 건) 전량을 시스템에 적재하는 단계 — 이후 모든 분석·모델링의 원천 데이터다.
mercari_df = pd.read_csv('../data/mercari_train.tsv', sep='\t')

# 기술적 의미: 방금 읽은 DataFrame의 (행, 열) 크기를 출력해 예상 규모(1,482,535행 x 8열)와 일치하는지 확인한다. 별도의 표본 추출 없이 이 DataFrame 전체를 그대로 이후 분석·학습에 사용한다.
# 업무적 의미: 데이터 건수는 분석 착수 전 가장 먼저 검수해야 할 사실 확인(sanity check)이다 — 건수가 다르면 파일 경로·구분자 설정이 잘못됐다는 신호다. (2026-08-26 변경: 기존에는 응답 속도를 위해 20,000행 무작위 표본만 썼으나, 이번 변경으로 표본 추출 단계를 제거하고 전체 1,482,535행 전량을 학습·평가 대상으로 삼도록 소스를 전면 수정했다 — 이에 따라 학습 소요시간이 크게 늘어날 수 있음에 유의한다.)
print('전체 행수:', mercari_df.shape[0])  # (1,482,535, 8)

# 기술적 의미: 수치 연산 라이브러리 numpy를 np라는 별칭으로 불러온다.
# 업무적 의미: 바로 다음 줄의 로그 변환(log1p)에 필요한 도구 — 통계적 변환 작업의 표준 라이브러리다.
import numpy as np

# 기술적 의미: price(가격) 컬럼에 log1p(=log(1+x)) 변환을 적용한 결과를 새 컬럼 price_log에 저장한다.
# 업무적 의미: 중고거래 가격은 소수의 초고가 상품 때문에 분포가 한쪽으로 심하게 치우쳐 있다(오른쪽 꼬리가 긴 분포) — 로그 변환으로 "가격이 10배 차이나는 두 상품"과 "가격이 1.1배 차이나는 두 상품"을 모델이 비슷한 비중으로 학습하게 만드는, 가격 예측 실무의 표준 관행이다.
mercari_df['price_log'] = np.log1p(mercari_df['price'])

# 기술적 의미: 데이터프레임의 첫 10행을 출력해 컬럼 구성과 실제 값의 모양을 눈으로 확인한다.
# 업무적 의미: 본격 분석 전 "실제 데이터가 기대한 형태(상품명·가격·카테고리 등)로 정상 적재됐는지" 육안으로 검수하는 실무 관행이다.
mercari_df.head(40)


전체 행수: 1482535


,train_id,name,item_condition_id,category_name,brand_name,price,shipping,item_description,price_log
0,0,MLB Cincinnati Reds T Shirt Size XL,3,Men/Tops/T-shirts,NaN,10.0,1,No description yet,2.397895
1,1,Razer BlackWidow Chroma Keyboard,3,Electronics/Computers & Tablets/Components & P...,Razer,52.0,0,This keyboard is in great condition and works ...,3.970292
2,2,AVA-VIV Blouse,1,Women/Tops & Blouses/Blouse,Target,10.0,1,Adorable top with a hint of lace and a key hol...,2.397895
3,3,Leather Horse Statues,1,Home/Home Décor/Home Décor Accents,NaN,35.0,1,New with tags. Leather horses. Retail for [rm]...,3.583519
4,4,24K GOLD plated rose,1,Women/Jewelry/Necklaces,NaN,44.0,0,Complete with certificate of authenticity,3.806662
5,5,Bundled items requested for Ruie,3,Women/Other/Other,NaN,59.0,0,"Banana republic bottoms, Candies skirt with ma...",4.094345
6,6,Acacia pacific tides santorini top,3,Women/Swimwear/Two-Piece,Acacia Swimwear,64.0,0,Size small but straps slightly shortened to fi...,4.174387
7,7,Girls cheer and tumbling bundle of 7,3,Sports & Outdoors/Apparel/Girls,Soffe,6.0,1,You get three pairs of Sophie cheer shorts siz...,1.945910
8,8,Girls Nike Pro shorts,3,Sports & Outdoors/Apparel/Girls,Nike,19.0,0,Girls Size small Plus green. Three shorts total.,2.995732
9,9,Porcelain clown doll checker pants VTG,3,Vintage & Collectibles/Collectibles/Doll,NaN,8.0,0,I realized his pants are on backwards after th...,2.197225


## 2. 전처리 — 원본 노트북(10번)과 동일한 피처 엔지니어링

```mermaid
flowchart LR
    A["category_name<br/>'대/중/소'로 분할"] --> E["피처 결합(hstack)"]
    B["name → CountVectorizer"] --> E
    C["item_description → TF-IDF(1~3gram)"] --> E
    D["brand_name/item_condition_id/<br/>대·중·소분류 → LabelBinarizer 원-핫"] --> E
    E --> F["X_all (sparse 피처 행렬)"]
    G["shipping(0/1)"] -.->|"피처 아님, Part B의 분류 타깃"| F
```

In [3]:
# 기술적 의미: category_name(예: "Women/Tops/Blouse")을 '/' 기준으로 나눠 리스트로 반환하는 함수를 정의한다. 값이 없거나(NaN) 문자열이 아니어서 .split()이 실패하면 ['Other_Null']*3을 반환한다.
# 업무적 의미: 카테고리는 "대분류/중분류/소분류" 3단계 계층 구조로 되어 있는데, 이를 각각 분리해야 "신발 카테고리는 가격이 높다" 같은 세분화된 업무 규칙을 모델이 배울 수 있다 — 결측 처리는 "카테고리 미기재 상품도 버리지 않고 별도 그룹으로 분석에 포함시킨다"는 데이터 품질 정책이다.
def split_cat(category_name):
    try:
        return category_name.split('/')
    except Exception:
        return ['Other_Null', 'Other_Null', 'Other_Null']

# 기술적 의미: apply()로 category_name 컬럼의 각 값에 split_cat()을 적용해 3개짜리 리스트를 만들고, zip(*...)으로 이를 대분류/중분류/소분류 3개의 개별 컬럼(cat_dae/cat_jung/cat_so)으로 분해해 데이터프레임에 추가한다.
# 업무적 의미: "전자기기 > 컴퓨터 > 노트북"처럼 세분화된 카테고리 정보를 모델이 각 계층별로 따로 활용할 수 있게 만드는 단계 — 대분류만 볼 때보다 훨씬 정교한 가격·배송 패턴을 포착할 수 있다.
mercari_df['cat_dae'], mercari_df['cat_jung'], mercari_df['cat_so'] = zip(
    *mercari_df['category_name'].apply(lambda x: split_cat(x))
)

# 기술적 의미: brand_name 컬럼의 결측치(NaN)를 문자열 'Other_Null'로 채운다.
# 업무적 의미: 브랜드를 기재하지 않은 개인 판매자 상품이 실제로 매우 많다(전체의 약 43%) — 이 자체가 "브랜드 없음"이라는 유의미한 신호이므로 삭제하지 않고 명시적인 카테고리 값으로 남긴다.
mercari_df['brand_name'] = mercari_df['brand_name'].fillna(value='Other_Null')

# 기술적 의미: category_name 컬럼의 결측치를 동일하게 'Other_Null'로 채운다.
# 업무적 의미: 카테고리 미기재 상품도 분석 대상에서 제외하지 않고 별도 그룹으로 취급해, 표본 손실 없이 전체 상품을 다룬다.
mercari_df['category_name'] = mercari_df['category_name'].fillna(value='Other_Null')

# 기술적 의미: item_description 컬럼의 결측치를 동일하게 'Other_Null'로 채운다.
# 업무적 의미: 설명글이 없는 상품(실제로 "No description yet" 같은 무성의한 등록도 많음)도 텍스트 피처 생성 단계에서 오류 없이 처리되도록 보장한다.
mercari_df['item_description'] = mercari_df['item_description'].fillna(value='Other_Null')

# 기술적 의미: 결측치 검증 결과를 출력하기 전 안내 문구를 먼저 표시한다.
# 업무적 의미: 이후 출력을 읽는 사람(리뷰어)에게 "지금부터는 결측치 검증 결과"임을 알려주는 문서화 습관이다.
print('결측치 확인(전부 0이어야 함):')

# 기술적 의미: 6개 컬럼 각각의 결측치(NaN) 개수를 세어 출력한다 — 전부 0이 나와야 fillna가 제대로 적용됐다는 뜻이다.
# 업무적 의미: "데이터 정제가 실제로 완료됐는가"를 코드로 직접 검증하는 품질 게이트다 — 이 확인 없이 다음 단계(벡터화)로 넘어가면 결측치로 인한 오류가 뒤늦게, 그리고 원인 파악이 어렵게 터질 수 있다.
print(mercari_df[['brand_name', 'category_name', 'item_description', 'cat_dae', 'cat_jung', 'cat_so']].isnull().sum())


결측치 확인(전부 0이어야 함):
brand_name          0
category_name       0
item_description    0
cat_dae             0
cat_jung            0
cat_so              0
dtype: int64


In [4]:
# 기술적 의미: 텍스트를 숫자 벡터로 바꿔주는 두 클래스(CountVectorizer, TfidfVectorizer)를 sklearn에서 불러온다.
# 업무적 의미: 머신러닝 모델은 문자열을 직접 이해하지 못하므로, "상품명"과 "상품설명" 같은 텍스트 정보를 가격·배송 예측에 활용하려면 반드시 이 변환 단계를 거쳐야 한다 — 텍스트 데이터를 정량적 신호로 바꾸는 핵심 단계다.
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# 기술적 의미: 단어 등장 횟수를 세는 CountVectorizer를 기본 설정(옵션 없음)으로 생성한다.
# 업무적 의미: 상품명은 문장이 짧고 핵심 단어(브랜드·모델명 등) 자체가 중요하므로, 단어의 "등장 여부·횟수"만 보는 단순한 방식으로 충분하다는 판단이다.
cnt_vec = CountVectorizer()

# 기술적 의미: name(상품명) 컬럼 전체에 CountVectorizer를 학습(fit)시키고 동시에 벡터로 변환(transform)해 sparse 행렬 X_name을 만든다.
# 업무적 의미: "아이폰", "나이키" 같은 상품명 속 단어 하나하나가 가격을 가늠하는 중요한 단서가 된다 — 실제 이 노트북의 회귀 결과(§A-3)에서도 상품명의 특정 브랜드 단어가 가격 예측에 크게 기여했다.
X_name = cnt_vec.fit_transform(mercari_df.name)

# 기술적 의미: item_description(상품설명)을 TF-IDF 방식으로 벡터화할 도구를 만든다. max_features=50000은 빈도 상위 5만 단어(구)로만 제한, ngram_range=(1,3)은 단어 하나부터 3단어 연속 구까지 피처로 사용, stop_words='english'는 the/a/is 같은 의미 없는 단어를 제거한다는 뜻이다.
# 업무적 의미: 상품설명은 문장이 길고 "정품", "박스 포함", "충전기 포함" 같은 표현이 가격에 큰 영향을 주는데, 단어 하나만 보면 이런 구(phrase)의 맥락을 놓친다 — 다만 전체 단어를 다 쓰면 계산이 느려지므로, 실무적으로 상위 5만 개로 제한해 속도와 정보량의 균형을 맞춘 것이다.
tfidf_descp = TfidfVectorizer(max_features=50000, ngram_range=(1, 3), stop_words='english')

# 기술적 의미: item_description 컬럼에 위 TF-IDF 벡터화기를 학습·적용해 sparse 행렬 X_descp를 만든다.
# 업무적 의미: 상품설명 텍스트를 가격 예측에 실제로 활용 가능한 형태로 변환하는 단계 — 원본 노트북(10번)의 실측(Ridge 기준 RMSLE 0.4984→0.4680)에서도 설명 텍스트를 포함하면 성능이 개선됨을 이미 확인한 바 있다.
X_descp = tfidf_descp.fit_transform(mercari_df['item_description'])

# 기술적 의미: name 벡터화 결과의 (행, 열) 크기를 출력한다 — 열 개수는 CountVectorizer가 학습한 고유 단어(어휘) 개수다.
# 업무적 의미: "상품명에 총 몇 개의 서로 다른 단어가 쓰였는가"를 확인하는 것으로, 이 어휘 규모가 곧 모델이 다룰 수 있는 상품명 표현의 다양성이다.
print('name 벡터화 shape:', X_name.shape)

# 기술적 의미: item_description 벡터화 결과의 (행, 열) 크기를 출력한다 — max_features=50000으로 제한했으므로 열 개수는 최대 50,000이다.
# 업무적 의미: 설명 텍스트에서 뽑아낸 피처 규모를 확인해, 설정한 상한(5만)이 실제로 적용됐는지 검증한다.
print('item_description 벡터화 shape:', X_descp.shape)

# 기술적 의미: 범주형(카테고리) 값을 0/1 원-핫 벡터로 바꿔주는 LabelBinarizer를 불러온다.
# 업무적 의미: "브랜드가 나이키인가 아닌가", "상품상태가 1등급인가"처럼 순서 없는 범주 정보를 모델이 이해할 수 있는 숫자 형태로 바꾸는 표준적인 방법이다.
from sklearn.preprocessing import LabelBinarizer

# 기술적 의미: brand_name 전용 LabelBinarizer를 만든다. sparse_output=True는 결과를 촘촘한 배열이 아니라 메모리 절약형 희소행렬로 만들라는 옵션이다.
# 업무적 의미: 브랜드 종류가 수천 개에 달해(원본 데이터 기준 4,810종) 이를 촘촘한 배열로 만들면 메모리를 과도하게 낭비한다 — 실무에서 대용량 범주형 데이터를 다룰 때 반드시 고려해야 하는 최적화다.
lb_brand_name = LabelBinarizer(sparse_output=True)

# 기술적 의미: brand_name 컬럼을 학습·변환해 브랜드별 원-핫 sparse 행렬 X_brand를 만든다.
# 업무적 의미: "이 상품이 어떤 브랜드인가"라는 정보를 가격·배송 예측 모델이 직접 활용할 수 있는 형태로 변환한다 — 회귀 결과(§A-3)에서 브랜드 미기재(Other_Null) 여부가 가격에 가장 큰 영향을 준 피처였다.
X_brand = lb_brand_name.fit_transform(mercari_df['brand_name'])

# 기술적 의미: item_condition_id(상품상태 1~5) 전용 LabelBinarizer를 만든다.
# 업무적 의미: 상품상태는 숫자(1~5)로 저장돼 있지만 실제로는 "매우 좋음~하자 있음"처럼 순서형 등급이라, 등급 간 산술적 거리(5-1=4)를 모델이 오해하지 않도록 범주형으로 다루는 편이 안전하다.
lb_item_cond_id = LabelBinarizer(sparse_output=True)

# 기술적 의미: item_condition_id 컬럼을 학습·변환해 상태등급별 원-핫 sparse 행렬 X_item_cond_id를 만든다.
# 업무적 의미: "상품 상태가 새 것에 가까운가, 하자가 있는가"는 중고거래 가격을 좌우하는 핵심 정보 중 하나다.
X_item_cond_id = lb_item_cond_id.fit_transform(mercari_df['item_condition_id'])

# 기술적 의미: cat_dae(대분류) 전용 LabelBinarizer를 만든다.
# 업무적 의미: "여성의류/전자기기/뷰티" 같은 최상위 카테고리 구분 — 카테고리별로 평균 가격대가 크게 다르므로 모델의 중요한 판단 기준이 된다.
lb_cat_dae = LabelBinarizer(sparse_output=True)

# 기술적 의미: cat_dae 컬럼을 학습·변환해 대분류별 원-핫 sparse 행렬 X_cat_dae를 만든다.
# 업무적 의미: 대분류 정보를 모델이 직접 활용 가능한 숫자 형태로 제공하는 단계다.
X_cat_dae = lb_cat_dae.fit_transform(mercari_df['cat_dae'])

# 기술적 의미: cat_jung(중분류) 전용 LabelBinarizer를 만든다.
# 업무적 의미: 대분류보다 한 단계 더 세분화된 카테고리(예: "여성의류 > 신발") — 실제 회귀 결과(§A-3)에서 중분류 "Shoes"·"Women's Handbags"가 가격 예측 상위 기여 피처로 나타났다.
lb_cat_jung = LabelBinarizer(sparse_output=True)

# 기술적 의미: cat_jung 컬럼을 학습·변환해 중분류별 원-핫 sparse 행렬 X_cat_jung을 만든다.
# 업무적 의미: 대분류만으로는 놓치는 세부 카테고리별 가격 패턴을 모델이 학습할 수 있게 한다.
X_cat_jung = lb_cat_jung.fit_transform(mercari_df['cat_jung'])

# 기술적 의미: cat_so(소분류) 전용 LabelBinarizer를 만든다.
# 업무적 의미: 카테고리 계층의 가장 세부 단계(예: "휴대폰 > 스마트폰") — 가장 구체적인 상품 종류 정보를 제공한다.
lb_cat_so = LabelBinarizer(sparse_output=True)

# 기술적 의미: cat_so 컬럼을 학습·변환해 소분류별 원-핫 sparse 행렬 X_cat_so를 만든다.
# 업무적 의미: 가장 세밀한 카테고리 신호까지 모델에 제공해, "휴대폰"이라는 소분류 자체가 가격에 미치는 영향(§A-3의 "Cell Phones & Smartphones")을 포착할 수 있게 한다.
X_cat_so = lb_cat_so.fit_transform(mercari_df['cat_so'])

# 기술적 의미: 여러 sparse 행렬을 가로 방향으로 이어붙이는 hstack 함수를 scipy.sparse에서 불러온다.
# 업무적 의미: 텍스트·브랜드·상태·카테고리 등 서로 다른 출처의 피처들을 모델 하나가 한 번에 입력받을 수 있는 단일 표(피처 행렬)로 통합하는 데 필요하다.
from scipy.sparse import hstack

# 기술적 의미: 지금까지 만든 7개의 피처 행렬(X_descp, X_name, X_brand, X_item_cond_id, X_cat_dae, X_cat_jung, X_cat_so)을 튜플로 묶는다. shipping은 의도적으로 포함하지 않는다(§2-4).
# 업무적 의미: "상품을 설명하는 모든 정보(텍스트+브랜드+상태+카테고리)를 한 곳에 모은다"는 피처 설계의 최종 목록을 코드로 명시한 것 — 이 목록 자체가 "가격·배송 여부를 예측하는 데 어떤 정보를 쓰기로 했는가"라는 업무적 의사결정의 기록이다.
sparse_matrix_list = (X_descp, X_name, X_brand, X_item_cond_id, X_cat_dae, X_cat_jung, X_cat_so)

# 기술적 의미: hstack으로 7개 행렬을 가로로 결합한 뒤 .tocsr()로 CSR(Compressed Sparse Row) 형식으로 변환한다 — 행 단위 접근(슬라이싱, train_test_split)에 효율적인 형식이다.
# 업무적 의미: Part A(회귀)와 Part B(분류) 두 모델이 공유할 "단일 진실 공급원(single source of truth)" 피처 행렬을 확정하는 단계 — 이후 두 모델은 이 X_all만 입력받는다.
X_all = hstack(sparse_matrix_list).tocsr()

# 기술적 의미: 최종 결합된 피처 행렬의 (행, 열) 크기를 출력한다.
# 업무적 의미: "이 노트북의 모델이 상품 하나당 몇 개의 정보(피처)를 보고 판단하는가"를 확인하는 것 — 이 숫자(62,845차원)가 이후 두 모델의 입력 규모를 결정한다.
print('최종 피처 행렬(shipping 제외) shape:', X_all.shape)


name 벡터화 shape: (1482535, 105757)
item_description 벡터화 shape: (1482535, 50000)
최종 피처 행렬(shipping 제외) shape: (1482535, 161568)


### 2-4. 왜 `shipping`을 피처에서 제외했는가

Part B(GradientBoostingClassifier)에서 `shipping`(배송비 구매자/판매자 부담, 0/1)을 **분류 정답(타깃)**으로 쓴다. 만약 `shipping`을 피처에도 그대로 넣으면, Part A(가격 회귀)에서는 문제가 없지만 **Part B에서는 모델이 정답을 피처로 그대로 받아보는 "정답 누출(data leakage)"**이 된다. 두 모델(Part A/B)이 정확히 같은 피처셋으로 공정하게 비교되도록, 이 노트북에서는 **Part A(회귀)에서도 일부러 `shipping`을 피처에서 제외**했다 — 이 때문에 회귀 성능이 `shipping`을 포함했을 때보다 약간 낮게 나올 수 있다(§3-3에서 재확인).

## Part A. GradientBoostingRegressor — 가격(연속값) 예측

### A-1. 배경과 사용 이유(요약)

GradientBoosting은 얕은 트리 여러 개를 순차적으로 쌓아, 각 트리가 이전까지의 예측 오차(잔차)를 보정하도록 학습하는 앙상블 기법이다(Friedman, 2001, "Greedy Function Approximation: A Gradient Boosting Machine"). `GradientBoostingRegressor`는 이 알고리즘을 **연속값 예측(회귀)** 에 적용한 것으로, 기본 손실함수는 `squared_error`(제곱오차)다. 이 노트북에서는 브랜드·카테고리·상품상태·상품명/설명 텍스트가 복합적으로 상호작용해 결정되는 **가격**을 예측하는 데 사용한다.

### A-2. 파라미터 설명

| 파라미터 | 값 | 의미 |
|---|---|---|
| `n_estimators` | 100 | 순차적으로 쌓을 트리 개수 |
| `max_depth` | 3 | 트리 1개의 최대 깊이(얕은 트리 여러 개 = Gradient Boosting의 철학) |
| `learning_rate` | 0.05 | 각 트리가 최종 예측에 기여하는 비율(축소율) |
| `random_state` | 156 | 재현성(이 노트북 시드 관례) |

In [ ]:
# 기술적 의미: sklearn에서 GradientBoostingRegressor(회귀용 Gradient Boosting 모델)를 불러온다.
# 업무적 의미: 이 노트북 Part A의 주인공 — 상품 정보로부터 "적정 가격"을 추정하는 핵심 알고리즘이다.
from sklearn.ensemble import GradientBoostingRegressor

# 기술적 의미: 데이터를 학습용/검증용으로 무작위 분할해주는 train_test_split을 불러온다.
# 업무적 의미: "모델이 학습에 쓰지 않은, 처음 보는 데이터에서도 잘 맞히는가"를 확인하려면 학습 데이터와 평가 데이터를 반드시 분리해야 한다 — 실무 모델 검증의 기본 원칙이다.
from sklearn.model_selection import train_test_split

# 기술적 의미: 회귀 모델의 설명력을 나타내는 결정계수(R²)를 계산하는 r2_score를 불러온다.
# 업무적 의미: RMSLE(오차 크기)만으로는 "이 모델이 얼마나 쓸만한가"를 직관적으로 전달하기 어렵다 — R²는 "가격 변동의 몇 %를 이 모델이 설명하는가"를 0~1(또는 그 이하) 값으로 보여줘 비전문가에게도 설명하기 쉽다.
from sklearn.metrics import r2_score

# 기술적 의미: 학습 소요시간을 측정하기 위한 표준 라이브러리 time을 불러온다.
# 업무적 의미: GradientBoost는 순차 알고리즘이라 느리다는 점(§Part A-1)을 "말로만" 설명하지 않고 실측 수치로 증명하기 위해 필요하다 — 화면 응답시간 설계와 직결되는 정보다.
import time

# 기술적 의미: RMSLE(Root Mean Squared Logarithmic Error)를 계산하는 함수를 정의한다. log가 아니라 log1p를 쓰는 이유는 가격이 0인 상품이 있어 log(0)의 -무한대를 피하기 위함이다.
# 업무적 의미: 가격처럼 값의 범위가 매우 넓은(1달러~2,000달러) 지표는 "절대 오차"보다 "비율 오차"로 채점하는 것이 실무적으로 더 합리적이다 — 이 지표는 캐글 원본 대회의 공식 평가지표이기도 하다.
def rmsle(y, y_pred):
    return np.sqrt(np.mean(np.power(np.log1p(y) - np.log1p(y_pred), 2)))

# 기술적 의미: log1p 공간에서 나온 예측값·정답을 expm1(로그 역변환)로 원래 가격 스케일로 되돌린 뒤 rmsle()를 호출하는 평가 함수를 정의한다.
# 업무적 의미: 모델은 log1p(price)를 학습·예측하지만, 실제 업무 의사결정(가격 제안)은 원래 달러 단위로 이뤄져야 하므로 평가 직전에 반드시 원 스케일로 복원해야 한다 — 이 복원을 빠뜨리면 채점 자체가 무의미해진다.
def evaluate_org_price(y_test, preds):
    preds_exmpm = np.expm1(preds)
    y_test_exmpm = np.expm1(y_test)
    return rmsle(y_test_exmpm, preds_exmpm)

# 기술적 의미: X_all(피처)과 price_log(타깃)를 80%(학습)/20%(검증)로 무작위 분할한다. random_state=156으로 분할 결과를 고정해 재현 가능하게 한다. 전체 데이터(1,482,535행) 기준 20%는 약 296,507건이다.
# 업무적 의미: 이 약 296,507건(전체 데이터의 20%)은 "실제 서비스에 배포했을 때 처음 만나는 신규 상품"을 흉내 낸 것이다 — 여기서 나온 성능이 이 모델의 실전 신뢰도를 대표한다.
X_train, X_test, y_train, y_test = train_test_split(
    X_all, mercari_df['price_log'], test_size=0.2, random_state=156
)

# 기술적 의미: GradientBoostingRegressor를 n_estimators=100(트리 100개)·max_depth=3(트리 깊이 3)·learning_rate=0.05(학습률)·random_state=156(재현성)으로 생성한다.
# 업무적 의미: 이 세 하이퍼파라미터 값은 실제 운영 대시보드(mlreg.py)와 동일하다 — "노트북에서 이해한 모델"과 "실제 서비스되는 모델"이 같은 설정임을 보장해, 이 노트북의 설명이 곧 운영 코드에 대한 설명이 되게 한다. (2026-08-26 참고: 이제 20,000행이 아닌 전체 1,482,535행을 학습하므로, 20,000행 표본 기준 실측치였던 학습 소요시간 약 61초보다 훨씬 오래 걸릴 수 있다 — 실제 소요시간은 fit_time 출력값으로 확인할 것.)
gbr = GradientBoostingRegressor(n_estimators=100, max_depth=3, learning_rate=0.05, random_state=156)

# 기술적 의미: 학습 시작 시각을 기록한다.
# 업무적 의미: 다음 줄(fit)의 실행 시간을 재기 위한 기준점 — "이 화면을 열었을 때 사용자가 몇 초를 기다려야 하는가"를 실측하기 위함이다.
start = time.time()

# 기술적 의미: 학습 데이터로 GradientBoostingRegressor를 실제로 학습(fit)시킨다 — 내부적으로 트리 100개를 순차적으로 쌓으며 잔차를 줄여나간다.
# 업무적 의미: 이 한 줄이 "상품 정보로부터 가격을 배우는" 이 노트북의 핵심 학습 단계다 — 여기서 만들어진 규칙이 이후 신규 상품의 가격 제안에 그대로 쓰인다.
gbr.fit(X_train, y_train)

# 기술적 의미: 학습 종료 시각에서 시작 시각을 빼 학습 소요시간(초)을 계산한다.
# 업무적 의미: "정확도를 위해 시간을 얼마나 썼는가"를 정량화 — 파라미터를 바꿔 정확도를 올리려는 시도가 응답시간에 어떤 대가를 요구하는지 판단하는 근거 자료가 된다.
fit_time = time.time() - start

# 기술적 의미: 학습된 모델로 검증 데이터(X_test)에 대한 예측값을 만든다 — 아직 log1p 공간의 값이다.
# 업무적 의미: "학습에 쓰지 않은 상품 약 296,507건(전체 데이터의 20%)에 대해 이 모델이 예측한 가격"을 만들어내는 단계 — 실제 서비스에서 신규 상품 가격을 제안하는 것과 동일한 절차다.
preds = gbr.predict(X_test)

# 기술적 의미: 예측값과 실제값을 evaluate_org_price()에 넘겨 RMSLE를 계산한다.
# 업무적 의미: "이 모델이 신규 상품 가격을 얼마나 정확히 맞히는가"를 하나의 숫자로 요약한 핵심 성과 지표(KPI)다.
rmsle_value = evaluate_org_price(y_test, preds)

# 기술적 의미: 예측값(log1p 공간)을 expm1로 원래 가격 스케일로 복원한다.
# 업무적 의미: R² 계산과 이후 해석을 "달러" 단위의 실감 나는 값으로 하기 위한 준비 단계다.
preds_price = np.expm1(preds)

# 기술적 의미: 실제 정답(log1p 공간)도 동일하게 expm1로 원래 가격 스케일로 복원한다.
# 업무적 의미: 예측값과 정답을 같은 스케일(달러)에서 비교해야 R²가 의미를 갖는다.
y_test_price = np.expm1(y_test)

# 기술적 의미: 원래 가격 스케일에서 R²(결정계수)를 계산한다.
# 업무적 의미: "가격의 변동성 중 몇 %를 이 모델이 설명하는가"를 나타내는 보조 지표 — RMSLE만으로는 와닿지 않는 "설명력"을 경영진·비전문가에게 전달할 때 유용하다.
r2 = r2_score(y_test_price, preds_price)

# 기술적 의미: 학습 소요시간을 소수점 1자리로 출력한다.
# 업무적 의미: 이 화면을 실제로 열었을 때 사용자가 체감할 대기시간의 실측치 — 응답속도 설계 판단의 근거다.
print(f'학습 소요시간: {fit_time:.1f}초')

# 기술적 의미: RMSLE 값을 소수점 4자리로 출력한다.
# 업무적 의미: 이 모델의 가격 예측 정확도를 대표하는 핵심 성과 지표를 사용자(리뷰어)가 바로 확인할 수 있게 한다.
print(f'RMSLE: {rmsle_value:.4f}')

# 기술적 의미: R² 값을 소수점 4자리로 출력한다.
# 업무적 의미: 정확도를 또 다른 관점(설명력)에서 보완적으로 제시해, 하나의 지표만으로 성능을 오판하지 않도록 돕는다.
print(f'R2(결정계수): {r2:.4f}')


In [ ]:
# 기술적 의미: 학습된 회귀 모델의 feature_importances_(피처별 중요도 배열)를 큰 값 순서로 정렬한 뒤, 상위 10개의 인덱스를 가져온다.
# 업무적 의미: "가격을 예측할 때 모델이 실제로 어떤 정보를 가장 중요하게 봤는가"를 확인하는 단계 — 모델을 블랙박스로 남기지 않고, 비즈니스 담당자가 납득할 수 있는 근거를 제시하기 위한 해석(interpretability) 작업이다.
top_idx = np.argsort(gbr.feature_importances_)[::-1][:10]

# 기술적 의미: 피처 행렬의 각 블록(설명 텍스트/상품명/브랜드/상태/카테고리)이 전체 인덱스 중 어디서 시작해 몇 칸을 차지하는지, hstack에 넣은 순서 그대로 목록으로 정의한다.
# 업무적 의미: 피처 중요도 배열은 "몇 번째 숫자 칸"만 알려줄 뿐 그것이 "어떤 단어·브랜드·카테고리인지"는 알려주지 않는다 — 이 목록은 숫자를 다시 사람이 이해할 수 있는 이름으로 되짚기 위한 지도(map)다.
blocks = [
    ('item_description(TF-IDF)', X_descp.shape[1], tfidf_descp.get_feature_names_out()),
    ('name(CountVec)', X_name.shape[1], cnt_vec.get_feature_names_out()),
    ('brand_name', X_brand.shape[1], lb_brand_name.classes_),
    ('item_condition_id', X_item_cond_id.shape[1], lb_item_cond_id.classes_),
    ('cat_dae', X_cat_dae.shape[1], lb_cat_dae.classes_),
    ('cat_jung', X_cat_jung.shape[1], lb_cat_jung.classes_),
    ('cat_so', X_cat_so.shape[1], lb_cat_so.classes_),
]

# 기술적 의미: 주어진 전역 인덱스가 blocks 목록의 어느 블록에 속하는지, 누적 오프셋을 더해가며 찾아 (블록이름, 실제 피처값)을 반환하는 함수를 정의한다.
# 업무적 의미: "62,271번 피처"라는 숫자를 "중분류=Shoes"라는, 사업 담당자가 바로 이해할 수 있는 말로 번역해주는 도구다.
def resolve_feature(idx):
    offset = 0
    for name, size, feat_names in blocks:
        if idx < offset + size:
            return name, feat_names[idx - offset]
        offset += size
    return '?', '?'

# 기술적 의미: 이어질 상위 10개 피처 출력 앞에 안내 문구를 표시한다.
# 업무적 의미: 결과를 읽는 사람에게 "지금부터 나오는 목록이 가격에 가장 큰 영향을 준 요인 순위"임을 알려준다.
print('가격 예측에 가장 크게 기여한 피처 top10:')

# 기술적 의미: top_idx의 각 인덱스에 대해 resolve_feature()로 이름을 찾고, 해당 인덱스의 중요도 값과 함께 한 줄씩 출력한다.
# 업무적 의미: "브랜드 미기재 여부", "특정 카테고리(신발·가방·휴대폰)", "설명 속 특정 단어(정품·박스·충전기)"처럼, 실제 가격에 영향을 주는 구체적 요인을 사업 언어로 제시하는 최종 결과물이다 — 마케팅·MD(상품기획) 조직이 바로 활용할 수 있는 인사이트다.
for i in top_idx:
    block, feat = resolve_feature(int(i))
    print(f'  중요도 {gbr.feature_importances_[i]:.4f} | {block} | "{feat}"')


가격 예측에 가장 크게 기여한 피처 top10:
  중요도 0.1487 | brand_name | "Other_Null"
  중요도 0.1216 | name(CountVec) | "lularoe"
  중요도 0.0927 | cat_jung | "Shoes"
  중요도 0.0685 | cat_jung | "Women's Handbags"
  중요도 0.0539 | item_description(TF-IDF) | "box"
  중요도 0.0494 | cat_so | "Cell Phones & Smartphones"
  중요도 0.0444 | item_description(TF-IDF) | "authentic"
  중요도 0.0264 | name(CountVec) | "bundle"
  중요도 0.0250 | cat_dae | "Beauty"
  중요도 0.0229 | brand_name | "Lululemon"


### A-3. 회귀 결과 해석

> ⚠ **2026-08-26 변경**: §1의 20,000행 무작위 표본 추출 로직을 제거하고 `mercari_train.tsv` 전체(1,482,535행)를 그대로 학습·평가에 사용하도록 소스를 변경했다. 아래 수치는 **변경 전 20,000행 표본 실행 결과(참고치)**이며, 전체 데이터로 재실행하면 RMSLE·R²·피처 중요도 순위가 달라질 수 있다 — 재실행 후 이 절의 수치를 새 실측값으로 교체해야 한다.

- **(20,000행 표본 참고치) RMSLE 0.6558, R² 0.1055** — 원본 노트북의 LightGBM(전체 148만 행, RMSLE 0.4564)보다 낮았다. 표본 크기 차이가 사라진 지금은 성능 격차의 원인을 (a) `shipping` 피처를 일부러 제외했다는 점(§2-4)과 (b) GradientBoost 기본 파라미터가 LightGBM만큼 튜닝되지 않았다는 점, 두 가지로 좁혀 재해석해야 한다.
- 아래 피처 중요도 해석(브랜드·카테고리·설명 텍스트)도 20,000행 표본 기준이다 — 전체 데이터 재실행 후 §A-2 코드의 top10 출력으로 실제 순위를 다시 확인할 것.
- **브랜드가 없는(`Other_Null`) 상품**이 가장 큰 영향을 준다 — 브랜드 미기재 상품은 가격이 체계적으로 낮게/높게 형성되는 경향이 있다는 뜻.
- 상품명에 **"lularoe"**(인기 의류 브랜드명이 브랜드 필드가 아니라 상품명 텍스트에 적힌 경우)가 있으면 가격에 큰 영향 — 정형 필드(`brand_name`)가 비어 있어도 텍스트(`name`)에서 같은 정보를 보완적으로 잡아낼 수 있음을 보여주는 사례.
- 중분류가 **Shoes / Women's Handbags / Computers & Tablets**, 소분류가 **Cell Phones & Smartphones**인 경우 — 카테고리 자체가 가격대를 크게 가른다(전자기기·신발/가방류가 가격 분산이 큰 카테고리).
- 설명글에 **"box"(박스 포함)**, **"authentic"(정품)**, **"charger"(충전기 포함)** 같은 단어가 있으면 가격에 영향 — 실제 중고거래에서 "정품 인증"·"부속품 포함" 여부가 가격에 반영된다는 상식과 일치한다.

## Part B. GradientBoostingClassifier — 배송비 부담 주체(shipping) 분류

### B-1. 배경과 사용 이유(요약)

`GradientBoostingClassifier`는 Part A와 **완전히 동일한 알고리즘(순차적 잔차 학습)**을 쓰지만, 손실함수가 회귀의 `squared_error` 대신 **로그 손실(log loss / deviance)**로 바뀐다 — 각 트리가 "오차값"이 아니라 "클래스일 확률의 오차(그레이디언트)"를 예측하도록 학습한다는 점이 핵심 차이다. 이 노트북에서는 `shipping`(0=구매자부담, 1=판매자부담/무료배송)을 분류 타깃으로 사용한다 — 인위적으로 만든 타깃이 아니라 원본 데이터에 실제로 존재하는 이진 컬럼이다.

### B-2. 파라미터 설명

Part A(회귀)와 **동일한 하이퍼파라미터 이름·값**을 쓰지만, 의미가 미묘하게 다르다.

| 파라미터 | 값 | 회귀(Part A)에서의 의미 | 분류(Part B)에서의 의미 |
|---|---|---|---|
| `n_estimators` | 100 | 잔차(오차값)를 보정하는 트리 개수 | 클래스 확률의 그레이디언트를 보정하는 트리 개수 |
| `max_depth` | 3 | 동일 | 동일 |
| `learning_rate` | 0.05 | 동일(축소율) | 동일(축소율) |
| `loss` | `squared_error`(기본) | 제곱오차 최소화 | `log_loss`(기본, 구 `deviance`) — 분류 확률의 로그손실 최소화 |

In [ ]:
# 기술적 의미: sklearn에서 GradientBoostingClassifier(분류용 Gradient Boosting 모델)와, 분류 성능을 채점하는 7가지 지표/도구 함수(정확도·AUC·재현율·F1·정밀도·분류 리포트·혼동행렬)를 불러온다.
# 업무적 의미: 이 노트북 Part B의 주인공 — 상품 정보로부터 "배송비를 구매자와 판매자 중 누가 부담하는가"를 예측하는 알고리즘과, "이 예측을 실제 업무에 써도 되는가"를 판단하는 데 필요한 accuracy/roc_auc/recall/f1_score 4대 핵심 지표를 포함한 평가 도구 일체다.
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, roc_auc_score, recall_score, f1_score, precision_score,
    classification_report, confusion_matrix,
)

# 기술적 의미: 분류의 정답(타깃)으로 shipping(0=구매자부담, 1=판매자부담) 컬럼을 지정한다.
# 업무적 의미: 인위적으로 만든 가짜 분류 문제가 아니라, 원본 데이터에 실제로 존재하는 "배송 정책" 정보를 예측 대상으로 삼은 것 — 실제로 판매자가 배송비 정책을 어떻게 정할지 참고할 수 있는 업무적 가치가 있다.
y_class = mercari_df['shipping']

# 기술적 의미: X_all(피처)과 y_class(타깃)를 80%/20%로 분할하되, stratify=y_class로 학습·검증 양쪽에 0/1 클래스 비율을 원본과 동일하게 유지한다.
# 업무적 의미: 분류 문제에서 한쪽 클래스가 검증셋에 너무 적게(혹은 많이) 배정되면 평가 자체가 왜곡된다 — stratify는 "공정한 성적표"를 보장하기 위한 표준 실무 절차다.
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X_all, y_class, test_size=0.2, random_state=156, stratify=y_class
)

# 기술적 의미: GradientBoostingClassifier를 Part A(회귀)와 동일한 하이퍼파라미터 값(n_estimators=100, max_depth=3, learning_rate=0.05, random_state=156)으로 생성한다.
# 업무적 의미: 회귀와 분류에서 같은 값을 써서, "같은 알고리즘을 문제 유형만 바꿔 적용했을 때의 차이"를 파라미터 차이가 아니라 순수하게 목적함수(손실함수) 차이로 비교할 수 있게 했다(§Part C).
gbc = GradientBoostingClassifier(n_estimators=100, max_depth=3, learning_rate=0.05, random_state=156)

# 기술적 의미: 분류 모델의 학습 시작 시각을 기록한다.
# 업무적 의미: 회귀(Part A)와 분류(Part B)의 학습 시간을 같은 방식으로 실측·비교하기 위한 기준점이다.
start = time.time()

# 기술적 의미: 학습 데이터로 GradientBoostingClassifier를 학습(fit)시킨다 — 내부적으로 트리 100개가 순차적으로 "클래스 확률의 그레이디언트"를 보정한다.
# 업무적 의미: "상품 정보로부터 배송비 부담 주체를 배우는" 이 노트북의 핵심 학습 단계다.
gbc.fit(Xc_train, yc_train)

# 기술적 의미: 분류 모델의 학습 소요시간을 계산한다.
# 업무적 의미: 회귀와 분류의 학습시간을 같은 표본·파라미터 조건에서 비교해(§Part C), GradientBoost의 학습비용이 손실함수 종류보다 트리 개수·깊이·표본 크기에 더 좌우된다는 것을 실측으로 보이기 위함이다.
fit_time_c = time.time() - start

# 기술적 의미: 학습된 모델로 검증 데이터에 대한 최종 예측 클래스(0 또는 1)를 만든다.
# 업무적 의미: "이 신규 상품은 무료배송일 것으로 예상되는가"를 최종적으로 한 가지 값으로 판정하는 결과다.
pred_class = gbc.predict(Xc_test)

# 기술적 의미: predict_proba()로 각 클래스일 확률을 구한 뒤, 클래스 1(판매자부담)일 확률만 골라 pred_proba에 저장한다.
# 업무적 의미: "예/아니오"라는 이분법적 판정 대신 "판매자가 부담할 확률이 몇 %인가"라는 정도(gradation) 정보를 남겨, 이후 임계값 조정이나 우선순위 정렬(예: 확률이 애매한 상품만 사람이 재검토) 같은 업무 활용이 가능해진다.
pred_proba = gbc.predict_proba(Xc_test)[:, 1]

# 기술적 의미: 예측 클래스와 실제 클래스를 비교해 정확도(전체 중 맞힌 비율)를 계산한다.
# 업무적 의미: "이 모델이 배송비 부담 주체를 얼마나 자주 맞히는가"를 가장 직관적으로 보여주는 지표다.
acc = accuracy_score(yc_test, pred_class)

# 기술적 의미: 예측 확률과 실제 클래스를 비교해 ROC-AUC(임계값에 무관한 판별력 지표, 0.5=무작위, 1.0=완벽)를 계산한다.
# 업무적 의미: 정확도는 임계값(보통 0.5) 하나에서만 측정되는 반면, AUC는 "확률 순위 자체가 얼마나 잘 매겨졌는가"를 보여줘 모델의 근본적인 판별력을 더 공정하게 평가한다.
auc = roc_auc_score(yc_test, pred_proba)

# 기술적 의미: recall_score(재현율)를 average='macro'로 계산한다 — 클래스 0/1 각각의 재현율을 구한 뒤 단순평균을 낸다(표본 수 차이를 가중하지 않음).
# 업무적 의미: "실제로 구매자부담인 상품·판매자부담인 상품을 각각 놓치지 않고 얼마나 잘 잡아내는가"를 두 클래스에 동일한 비중으로 요약한 값이다 — 이 값이 accuracy보다 낮게 나오면(§B-3에서 실측 확인) 클래스 간 성능 격차가 있다는 신호다.
recall_macro = recall_score(yc_test, pred_class, average='macro')

# 기술적 의미: f1_score(정밀도와 재현율의 조화평균)를 average='macro'로 계산한다.
# 업무적 의미: "잘못 놓치는 것(재현율)"과 "잘못 잡아내는 것(정밀도)"을 동시에 고려한 균형 지표 — 클래스 불균형(구매자부담 2,228건 vs 판매자부담 1,772건) 상황에서 accuracy 하나만 보고 성능을 오판하지 않도록 보완한다.
f1_macro = f1_score(yc_test, pred_class, average='macro')

# 기술적 의미: precision_score(정밀도)도 참고용으로 macro 평균을 계산해 둔다.
# 업무적 의미: "판매자부담이라고 예측한 것 중 실제로 맞은 비율"을 함께 봐야 recall/f1의 의미가 온전히 해석된다 — 4대 핵심 지표(accuracy/roc_auc/recall/f1)를 요청받았지만, 세 지표(recall/f1/precision)는 항상 함께 해석해야 하는 한 세트이므로 같이 계산해 둔다.
precision_macro = precision_score(yc_test, pred_class, average='macro')

# 기술적 의미: 학습 소요시간을 출력한다.
# 업무적 의미: Part A(61.1초)와 나란히 비교할 수 있는 실측 자료를 남긴다.
print(f'학습 소요시간: {fit_time_c:.1f}초')

# 기술적 의미: '핵심 지표 요약'이라는 안내 문구를 출력해, 이후 4줄이 이번에 요청받은 accuracy/roc_auc/recall/f1_score 지표임을 명확히 구분해 보여준다.
# 업무적 의미: 여러 출력이 뒤섞이지 않도록, "업무에서 바로 확인해야 할 4대 지표"만 모아 별도 섹션으로 강조하는 보고서 작성 관행이다.
print('--- 핵심 지표 요약(accuracy / roc_auc / recall / f1_score) ---')

# 기술적 의미: 정확도를 소수점 4자리로 출력한다.
# 업무적 의미: 이 분류 모델의 핵심 성과 지표를 바로 확인할 수 있게 한다.
print(f'Accuracy(정확도): {acc:.4f}')

# 기술적 의미: ROC-AUC를 소수점 4자리로 출력한다.
# 업무적 의미: 정확도만으로는 드러나지 않는 클래스별 판별력을 보완적으로 제시한다.
print(f'ROC-AUC222: {auc:.4f}')

# 기술적 의미: macro 평균 재현율을 소수점 4자리로 출력한다.
# 업무적 의미: "두 클래스를 놓치지 않고 잡아내는 능력"을 균형 있게 요약해, 특정 클래스만 잘 맞히는 편향된 모델이 아닌지 바로 확인할 수 있게 한다.
print(f'Recall(재현율, macro 평균): {recall_macro:.4f}')

# 기술적 의미: macro 평균 F1-score를 소수점 4자리로 출력한다.
# 업무적 의미: 정밀도·재현율을 동시에 고려한 종합 성능 지표를 한눈에 확인할 수 있게 한다 — 이 네 줄(accuracy/roc_auc/recall/f1_score)이 이번 요청에서 확인이 필요하다고 지목된 지표들이다.
print(f'F1-score(macro 평균): {f1_macro:.4f}')

# 기술적 의미: 참고용으로 macro 평균 정밀도도 함께 출력한다.
# 업무적 의미: recall/f1만 보면 "정밀도가 얼마나 희생됐는지"를 알 수 없다 — 세 지표를 나란히 둬야 트레이드오프를 정확히 판단할 수 있다.
print(f'Precision(정밀도, macro 평균, 참고): {precision_macro:.4f}')

# 기술적 의미: 출력 사이에 빈 줄을 하나 넣어 가독성을 높인다.
# 업무적 의미: 보고서·화면 형태로 옮겨질 때도 읽기 편하도록 구성하는 습관이다.
print()

# 기술적 의미: 각 클래스(구매자부담/판매자부담)별 precision(정밀도)·recall(재현율)·f1-score와 표본 수(support)를 표 형태로 출력한다.
# 업무적 의미: 위 macro 평균만으로는 "어느 클래스가 더 취약한가"가 안 보인다 — 전체 정확도 하나만 보면 "판매자부담 클래스를 유독 잘 못 맞힌다"는 사실(§B-3, recall 0.49)이 가려지므로, 클래스별 상세 성적표는 실제 서비스 적용 전 반드시 확인해야 할 정보다.
print(classification_report(yc_test, pred_class, target_names=['구매자부담(0)', '판매자부담(1)']))

# 기술적 의미: 혼동행렬(실제 클래스 x 예측 클래스 교차표) 출력 전 안내 문구를 표시한다.
# 업무적 의미: 이어질 표가 "실제로는 어떤데 모델은 어떻게 예측했는가"의 구체적 사례 건수임을 알려준다.
print('혼동행렬(행=실제, 열=예측):')

# 기술적 의미: 실제 클래스와 예측 클래스로 2x2 혼동행렬을 만들어 출력한다.
# 업무적 의미: "판매자부담 상품을 구매자부담으로 잘못 예측한 건수가 유독 많다"(§B-3) 같은, 정확도 숫자 하나로는 안 보이는 오류의 방향성과 패턴을 구체적으로 드러낸다 — 개선 방향(예: class_weight 조정)을 정하는 근거 자료다.
print(confusion_matrix(yc_test, pred_class))


학습 소요시간: 14065.9초
--- 핵심 지표 요약(accuracy / roc_auc / recall / f1_score) ---
Accuracy(정확도): 0.6707
ROC-AUC222: 0.7123
Recall(재현율, macro 평균): 0.6533
F1-score(macro 평균): 0.6517
Precision(정밀도, macro 평균, 참고): 0.6745

              precision    recall  f1-score   support

    구매자부담(0)       0.66      0.82      0.73    163887
    판매자부담(1)       0.68      0.49      0.57    132620

    accuracy                           0.67    296507
   macro avg       0.67      0.65      0.65    296507
weighted avg       0.67      0.67      0.66    296507

혼동행렬(행=실제, 열=예측):
[[134074  29813]
 [ 67831  64789]]


### B-3. 분류 결과 해석

> ⚠ **2026-08-26 변경**: §1의 20,000행 무작위 표본 추출 로직을 제거하고 전체 데이터(1,482,535행)를 사용하도록 변경했다. 아래 표·수치는 **변경 전 20,000행 표본 실행 결과(참고치)**이며, 전체 데이터 재실행 후 새 값으로 갱신해야 한다.

**핵심 지표 4종(20,000행 표본 참고치)**

| 지표 | 값 | 의미 |
|---|---|---|
| Accuracy(정확도) | 0.6723 | 전체 검증 표본 중 배송비 부담 주체를 맞힌 비율 |
| ROC-AUC | 0.7135 | 임계값에 무관한 판별력(0.5=무작위, 1.0=완벽) — 무작위 추측보다 뚜렷하게 나은 판별력 |
| Recall(재현율, macro 평균) | 0.6535 | 두 클래스(구매자부담/판매자부담)를 각각 놓치지 않고 잡아내는 능력의 평균 |
| F1-score(macro 평균) | 0.6524 | 정밀도·재현율을 동시에 고려한 종합 지표의 평균 |
| (참고) Precision(정밀도, macro 평균) | 0.6746 | recall/f1과 항상 함께 해석해야 하는 지표라 참고용으로 같이 표기 |

- **Recall(macro, 0.6535)이 Accuracy(0.6723)보다 낮다** — 이는 클래스 불균형(구매자부담 2,228건 vs 판매자부담 1,772건, 20,000행 표본 기준)의 전형적인 신호다. 클래스별로 뜯어보면 구매자부담(0)의 recall은 0.82로 높지만 판매자부담(1)의 recall은 0.49에 불과해, **macro 평균 하나만 보면 안 되고 반드시 클래스별 값(아래 classification_report)까지 함께 확인해야 한다.**
- **F1-score(macro, 0.6524)**도 같은 이유로 accuracy보다 낮다 — recall이 끌어내린 결과다. precision(macro, 0.6746)은 상대적으로 높아, "틀리게 판매자부담이라고 예측하는 경우"보다 "실제 판매자부담인데 놓치는 경우"가 더 많은 모델임을 알 수 있다.
- **정확도 0.6723, ROC-AUC 0.7135** — 무작위 추측(AUC 0.5)보다 뚜렷하게 나은 판별력을 보인다. 상품명·설명·카테고리·브랜드 정보만으로 "배송비를 누가 부담하는가"를 어느 정도 맞힐 수 있다는 뜻이다(예: 저가 상품일수록 판매자가 배송비를 부담해 무료배송으로 표시하는 경향 등, 실제 이커머스 관행과도 부합).
- **구매자부담(0) 클래스의 recall(0.82)이 판매자부담(1) 클래스의 recall(0.49)보다 높다** — 이 표본에서 구매자부담(0) 건수가 더 많아(support 2228 vs 1772) 모델이 다수 클래스 쪽으로 조금 더 치우쳐 예측하는 경향을 보인다. 실제 서비스라면 `class_weight='balanced'` 같은 옵션을 검토할 만한 지점이다.
- 혼동행렬을 보면 오분류가 406건(실제 0을 1로 잘못 예측)+905건(실제 1을 0으로 잘못 예측)=**1,311건** — 전체 4,000건(20,000행 표본의 20%) 중 약 32.8%로, 정확도(0.6723 → 오류율 32.77%)와 정확히 일치한다. 특히 "실제 1인데 0으로 잘못 예측"한 905건이 압도적으로 많다 — 이는 앞서 언급한 판매자부담(1) 클래스의 낮은 recall(0.49)과 같은 현상을 다른 각도(혼동행렬)에서 재확인한 것이다. 전체 데이터(약 296,507건 검증)로 재실행하면 이 절대 건수들은 모두 바뀐다.

## Part C. GradientBoostingRegressor vs GradientBoostingClassifier — 같은 알고리즘, 다른 목적함수

```mermaid
flowchart TD
    A["공통 피처 파이프라인<br/>(name/description/brand/condition/category)"] --> B["GradientBoostingRegressor<br/>타깃: price(연속값)"]
    A --> C["GradientBoostingClassifier<br/>타깃: shipping(0/1 이진)"]

    B --> D["손실함수: squared_error<br/>각 트리 = 잔차(오차값) 예측"]
    C --> E["손실함수: log_loss(deviance)<br/>각 트리 = 클래스 확률의 그레이디언트 예측"]

    D --> F["평가: RMSLE, R²<br/>(20,000행 표본 참고치) RMSLE=0.6558, R2=0.1055"]
    E --> G["평가: Accuracy, ROC-AUC<br/>(20,000행 표본 참고치) ACC=0.6723, AUC=0.7135"]

    style B fill:#E4F5E9,stroke:#2E8B57
    style C fill:#E0ECFF,stroke:#2E6DE5
```

> ⚠ 위 두 참고치와 아래 표의 실측 학습시간은 **20,000행 표본 기준(변경 전)**이다. 2026-08-26에 §1의 표본 추출 로직을 제거하고 전체 데이터(1,482,535행)를 쓰도록 변경했으므로, 전체 데이터로 재실행하면 모든 수치가 갱신되어야 한다.

| 구분 | GradientBoostingRegressor | GradientBoostingClassifier |
|---|---|---|
| 타깃 | 연속값(price, log1p) | 이산값(shipping, 0/1) |
| 기본 손실함수 | `squared_error` | `log_loss`(구 `deviance`) |
| 트리가 학습하는 대상 | 잔차(실제값-예측값) | 클래스 확률의 그레이디언트 |
| 주요 평가지표 | RMSLE, R², MAE | Accuracy, ROC-AUC, F1 |
| `predict()` 반환값 | 연속 예측값 | 클래스 레이블(0/1) |
| `predict_proba()` | 없음(회귀는 확률 개념이 없음) | 있음 — 클래스별 확률 |
| 20,000행 표본 참고 학습시간 | 61.1초 | 61.3초 (동일 표본·파라미터 조건에서 거의 동일 — 전체 데이터 기준 시간은 재실행 후 확인) |

## Part D. 내부테스트(자체 검증)

이 노트북은 FastAPI 백엔드(`semi-backend`)처럼 pytest를 쓰는 서비스 코드가 아니라 **분석용 노트북**이므로, "내부테스트"는 앞선 계산 결과가 **상식적으로 말이 되는 범위 안에 있는지**를 코드로 직접 검증하는 방식으로 수행한다. 아래 셀의 모든 `assert`가 예외 없이 통과하면 이 노트북의 계산 결과가 유효하다는 뜻이다.

```mermaid
flowchart TD
    A["TC-1: 피처 행렬 행수 = 표본 행수"] --> H{"전부 통과?"}
    B["TC-2: RMSLE ≥ 0"] --> H
    C["TC-3: R² ≤ 1"] --> H
    D["TC-4: Accuracy·AUC가 0~1 범위"] --> H
    E["TC-5: 혼동행렬 합계 = 검증 표본 수"] --> H
    F["TC-6: 예측값에 NaN/inf 없음"] --> H
    G["TC-7(신규): Recall·F1-score(macro)가 0~1 범위"] --> H
    H -- "예" --> I["✅ 내부테스트 통과"]
    H -- "아니오" --> J["❌ AssertionError로 즉시 중단"]

    style G fill:#FFF3CD,stroke:#B8860B
    style I fill:#E4F5E9,stroke:#2E8B57
    style J fill:#FFE0E0,stroke:#C0392B
```

In [ ]:
# 기술적 의미: 피처 행렬 X_all의 행 개수가 원본 표본 데이터프레임 mercari_df의 행 개수와 정확히 같은지 확인한다. 다르면 AssertionError를 발생시켜 즉시 실행을 중단한다.
# 업무적 의미: 이 값이 어긋나면 "어떤 상품의 피처와 다른 상품의 정답이 잘못 짝지어지는" 가장 치명적인 버그가 이미 발생했다는 뜻이다 — 이후의 모든 학습·평가 결과 자체가 신뢰할 수 없게 되므로, 가장 먼저 확인해야 할 검증이다.
assert X_all.shape[0] == len(mercari_df), 'TC-1 실패: 피처 행렬 행수가 mercari_df와 다름'

# 기술적 의미: RMSLE는 정의상(오차를 제곱한 뒤 평균 내고 제곱근을 취하는 구조) 항상 0 이상이어야 하므로, 이를 코드로 재확인한다.
# 업무적 의미: 오차 지표가 음수로 나온다면 계산식 자체에 버그가 있다는 뜻이므로, "성능 수치를 그대로 믿어도 되는가"에 대한 최소한의 안전장치다.
assert rmsle_value >= 0, 'TC-2 실패: RMSLE가 음수'

# 기술적 의미: R²(결정계수)는 정의상 1을 초과할 수 없으므로(완벽 예측일 때 최댓값이 1) 이를 재확인한다.
# 업무적 의미: R²가 1을 넘는다면 계산 과정(예: 스케일 불일치)에 오류가 있다는 신호이며, "설명력 지표를 경영진에게 그대로 보고해도 되는가"를 사전에 검증한다.
assert r2 <= 1.0, 'TC-3 실패: R2가 1을 초과함'

# 기술적 의미: 정확도(Accuracy)는 정의상 0~1 사이 값이어야 하므로 이를 재확인한다.
# 업무적 의미: 분류 성능 지표가 정상 범위를 벗어나면 평가 코드 자체에 문제가 있다는 뜻이므로, 결과를 보고하기 전 마지막 방어선이다.
assert 0.0 <= acc <= 1.0, 'TC-4 실패: Accuracy가 0~1 범위를 벗어남'

# 기술적 의미: ROC-AUC 역시 정의상 0~1 사이 값이어야 하므로 이를 재확인한다.
# 업무적 의미: AUC가 범위를 벗어난다면 확률값 계산(predict_proba)이나 클래스 라벨 처리에 오류가 있다는 신호다.
assert 0.0 <= auc <= 1.0, 'TC-4 실패: AUC가 0~1 범위를 벗어남'

# 기술적 의미: 실제·예측 클래스로 혼동행렬을 다시 계산해 cm 변수에 저장한다.
# 업무적 의미: 아래 검증(TC-5)에 쓸 값을 준비하는 단계이며, 위(§Part B)에서 이미 출력한 혼동행렬과 동일한 계산을 한 번 더 수행해 일관성을 교차 확인하는 효과도 있다.
cm = confusion_matrix(yc_test, pred_class)

# 기술적 의미: 혼동행렬 전체 원소의 합(=전체 예측 건수)이 분류 검증 표본 수(len(yc_test))와 정확히 같은지 확인한다.
# 업무적 의미: 일부 상품이 누락되거나 중복 집계되지 않았는지, 즉 "전수를 빠짐없이 채점했는가"를 검증하는 항목이다 — 실무 보고서에서 흔히 발생하는 "합계가 안 맞는" 오류를 코드로 미리 차단한다.
assert cm.sum() == len(yc_test), 'TC-5 실패: 혼동행렬 합계가 검증 표본 수와 다름'

# 기술적 의미: 회귀 예측값(preds) 전부가 유한한 값(NaN도 무한대도 아님)인지 확인한다.
# 업무적 의미: sparse 행렬 처리 과정에서 흔히 발생하는 실수(0으로 나누기, 빈 벡터 등)로 예측값이 깨지는 경우를 조기에 잡아내는, "실제 서비스에 내보내기 전 마지막 품질 검사"다.
assert np.isfinite(preds).all(), 'TC-6 실패: 회귀 예측값에 NaN 또는 inf 존재'

# 기술적 의미: 분류 확률 예측값(pred_proba) 전부가 유한한 값인지 확인한다.
# 업무적 의미: 확률값에 결측이 있으면 이후 AUC 계산이나 우선순위 정렬 같은 후속 업무 활용이 불가능해지므로, 이를 사전에 방지한다.
assert np.isfinite(pred_proba).all(), 'TC-6 실패: 분류 확률 예측값에 NaN 또는 inf 존재'

# TC-7: recall_macro와 f1_macro도 정의상 반드시 0~1 사이 값이어야 한다.
# 기술적 의미: 이번에 새로 추가한 recall_score/f1_score(macro 평균) 결과가 정상 범위인지 확인한다.
# 업무적 의미: 이번 요청으로 새로 노출한 4대 지표(accuracy/roc_auc/recall/f1_score) 전부가 검증 대상에 포함되도록, 자체 테스트 범위를 함께 확장했다 — 새 기능을 추가하면서 검증을 빠뜨리지 않는 것이 실무 품질관리의 기본이다.
assert 0.0 <= recall_macro <= 1.0, 'TC-7 실패: Recall(macro)이 0~1 범위를 벗어남'
assert 0.0 <= f1_macro <= 1.0, 'TC-7 실패: F1-score(macro)가 0~1 범위를 벗어남'

# 기술적 의미: 위 7개 검증(TC-1~TC-7)이 모두 통과했다는 안내 문구를 출력한다.
# 업무적 의미: 이 노트북의 모든 수치가 "코드로 재검증된, 신뢰할 수 있는 결과"임을 최종 확인해주는 결론 메시지다.
print('TC-1 ~ TC-7 전부 통과 — 내부테스트 성공')

# 기술적 의미: 피처 행렬 크기와 표본 행수를 요약해 출력한다.
# 업무적 의미: 이 실행이 어떤 규모의 데이터를 대상으로 이뤄졌는지 결과와 함께 기록해, 나중에 다시 볼 때도 맥락을 알 수 있게 한다.
print(f'  - 피처 행렬: {X_all.shape}, 표본: {len(mercari_df)}행')

# 기술적 의미: 회귀 성능(RMSLE, R2)을 요약해 출력한다.
# 업무적 의미: Part A의 핵심 결과를 검증 로그 안에도 함께 남겨, 결과서 작성 시 이 한 줄만 봐도 회귀 성능을 알 수 있게 한다.
print(f'  - 회귀: RMSLE={rmsle_value:.4f}, R2={r2:.4f}')

# 기술적 의미: 분류 성능 4대 지표(Accuracy/AUC/Recall/F1)와 혼동행렬 합계를 요약해 출력한다.
# 업무적 의미: Part B의 핵심 결과(이번에 요청받은 accuracy/roc_auc/recall/f1_score 전부)를 검증 로그 안에도 함께 남겨, 이 노트북 하나로 회귀·분류 두 모델의 최종 성적을 한눈에 확인할 수 있게 한다.
print(f'  - 분류: Accuracy={acc:.4f}, AUC={auc:.4f}, Recall(macro)={recall_macro:.4f}, F1(macro)={f1_macro:.4f}, 혼동행렬 합계={cm.sum()}')


TC-1 ~ TC-7 전부 통과 — 내부테스트 성공
  - 피처 행렬: (1482535, 161568), 표본: 1482535행
  - 회귀: RMSLE=0.6535, R2=0.0906
  - 분류: Accuracy=0.6707, AUC=0.7123, Recall(macro)=0.6533, F1(macro)=0.6517, 혼동행렬 합계=296507


## 결론 요약

- `mercari_train.tsv` 데이터를 기준으로, **GradientBoostingRegressor(가격 예측)와 GradientBoostingClassifier(배송비 부담 분류)** 두 모듈을 하나의 공통 피처 파이프라인 위에서 각각 실제로 학습·평가했다.
- **2026-08-26 변경**: §1의 20,000행 무작위 표본 추출 로직을 제거하고, `mercari_train.tsv` 전체(1,482,535행)를 표본 없이 그대로 학습·평가 대상으로 사용하도록 소스 전체를 변경했다.
- (20,000행 표본 참고치, 변경 전) 회귀: RMSLE 0.6558 / R² 0.1055 (`shipping` 피처 제외 조건).
- (20,000행 표본 참고치, 변경 전) 분류(핵심 지표 4종): **Accuracy 0.6723 / ROC-AUC 0.7135 / Recall(macro) 0.6535 / F1-score(macro) 0.6524** (동일 표본·피처 조건) — 클래스별 상세값(구매자부담 recall 0.82, 판매자부담 recall 0.49)은 §B-3 참고.
- 위 참고치들은 전체 데이터로 재실행하기 전 값이므로, **전체 데이터 재실행 후 §A-3/§B-3/§Part C/§Part E의 수치를 실측값으로 반드시 교체해야 한다** — 학습 소요시간도 20,000행 기준 약 61초보다 대폭 늘어날 수 있음에 유의한다.
- 두 모듈은 같은 Gradient Boosting 알고리즘이지만 손실함수(제곱오차 vs 로그손실)와 평가지표(RMSLE/R² vs Accuracy/AUC/Recall/F1)가 다르다는 것을 실제 코드 실행으로 확인했다.
- Part D의 자체 검증(TC-1~TC-7)은 특정 표본 크기에 의존하지 않는 구조적 검증(범위·정합성 확인)이므로, 전체 데이터로 재실행해도 그대로 유효하다.
- 상세한 GradientBoost 배경·장단점·파라미터 심화 설명은 `10 캐글 mercari price_정제후.ipynb` §6과 `95.작업결과문서/GradientBoost_모델_배경과파라미터_분석_20260825122306.md`에 이미 정리되어 있어 이 노트북에서는 중복 서술하지 않았다.

## Part F 이전 — GradientBoost 외 4개 모델(XGBoost/LightGBM/LinearRegression/LogisticRegression) 비교 실습

이 지점부터는 위 Part A~D(GradientBoost, 전체 데이터 1,482,535행 기준 실행 완료 — §Part A-3 RMSLE 0.6535/R² 0.0906, §Part B-3 Accuracy 0.6707/AUC 0.7123)에서 만든 **동일한 피처 행렬(`X_all`)과 동일한 학습/검증 분할(`X_train`/`X_test`/`y_train`/`y_test`, `Xc_train`/`Xc_test`/`yc_train`/`yc_test`)을 그대로 재사용**해, 나머지 4개 모델(XGBoost, LightGBM, LinearRegression, LogisticRegression)을 "GradientBoost와 같은 조건"에서 돌려본다. 새로 피처를 만들거나 데이터를 다시 나누지 않는다 — 그래야 5개 모델의 성능 차이가 순수하게 **알고리즘 자체의 차이**에서 온 것이라고 해석할 수 있다.

| 모델 | 회귀(price) | 분류(shipping) | 비고 |
|---|---|---|---|
| ① GradientBoost | ✅ Part A(위, 완료) | ✅ Part B(위, 완료) | 이 노트북의 기준(baseline) |
| ② XGBoost | ✅ Part F | ✅ Part F | GradientBoost와 같은 트리 개수·깊이·학습률 |
| ③ LightGBM | ✅ Part G | ✅ Part G | 동일 조건 + 리프 단위 성장 특성상 `num_leaves`로 깊이 상한을 추가 고정 |
| ④ LinearRegression | ✅ Part H | — (분류 모델 아님) | 트리 기반이 아닌 선형모델 기준점(baseline) |
| ⑤ LogisticRegression | — (회귀 모델 아님) | ✅ Part I | 트리 기반이 아닌 선형모델 기준점(baseline) |

> 회귀·분류를 모두 지원하는 그레이디언트 부스팅 3종(GradientBoost/XGBoost/LightGBM)은 동일한 하이퍼파라미터(`n_estimators=100, max_depth=3, learning_rate=0.05, random_state=156`)로 맞춰 "같은 조건"을 유지했다. LinearRegression/LogisticRegression은 알고리즘 특성상 트리 개수 같은 파라미터가 없으므로, 이 노트북 관례(`random_state=156`, 가능한 경우)만 맞추고 각자의 실무 표준 설정을 사용했다(§H-1, §I-1에서 설명).

```mermaid
flowchart TD
    subgraph SHARED["공통 준비물 (Part 1~2, 위에서 이미 실행됨)"]
        X["X_all 피처 행렬<br/>(1,482,535 x 161,568, sparse)"]
        SPLIT_R["회귀 분할<br/>X_train/X_test/y_train/y_test"]
        SPLIT_C["분류 분할<br/>Xc_train/Xc_test/yc_train/yc_test"]
        X --> SPLIT_R
        X --> SPLIT_C
    end

    SPLIT_R --> M1R["① GradientBoostingRegressor<br/>(Part A, 완료: RMSLE 0.6535)"]
    SPLIT_C --> M1C["① GradientBoostingClassifier<br/>(Part B, 완료: AUC 0.7123)"]

    SPLIT_R --> M2R["② XGBRegressor (Part F)"]
    SPLIT_C --> M2C["② XGBClassifier (Part F)"]

    SPLIT_R --> M3R["③ LGBMRegressor (Part G)"]
    SPLIT_C --> M3C["③ LGBMClassifier (Part G)"]

    SPLIT_R --> M4R["④ LinearRegression (Part H)"]

    SPLIT_C --> M5C["⑤ LogisticRegression (Part I)"]

    M1R --> CMP["Part J. 5개 모델 종합 비교<br/>(RMSLE/R² · Accuracy/AUC/Recall/F1 · 학습시간)"]
    M2R --> CMP
    M3R --> CMP
    M4R --> CMP
    M1C --> CMP
    M2C --> CMP
    M3C --> CMP
    M5C --> CMP

    CMP --> TEST["Part K. 신규 모델 내부테스트<br/>(TC-12 ~ TC-15)"]

    style M1R fill:#E4F5E9,stroke:#2E8B57
    style M1C fill:#E4F5E9,stroke:#2E8B57
    style CMP fill:#FFF3CD,stroke:#B8860B
    style TEST fill:#E0ECFF,stroke:#2E6DE5
```

> ⚠ **실행 상태 안내**: 아래 Part F~K는 이번에 새로 추가한 소스로, 이 시점까지는 **아직 실행하지 않았다**. 각 코드 셀은 위 Part A~B와 정확히 같은 변수(`X_train`, `X_test`, `y_train`, `y_test`, `Xc_train`, `Xc_test`, `yc_train`, `yc_test`, `rmsle()`, `evaluate_org_price()`, `blocks`, `resolve_feature()`)를 재사용하므로, **반드시 Part A·B가 먼저 실행된 뒤에(즉 이 노트북을 처음부터 순서대로 실행할 때) 정상 동작한다.**

## Part F. XGBoost — 가격(회귀) / 배송비(분류)

### F-1. 배경과 사용 이유(요약)

XGBoost(eXtreme Gradient Boosting)는 GradientBoost와 같은 "얕은 트리를 순차적으로 쌓아 잔차를 보정"하는 그레이디언트 부스팅 계열이지만, (a) 손실함수에 **정규화항(트리 복잡도 벌점)**을 추가해 과적합을 억제하고, (b) **히스토그램 기반 분할 탐색**으로 대용량 데이터에서 sklearn의 `GradientBoostingRegressor`/`Classifier`보다 훨씬 빠르게 학습한다는 점이 핵심 차이다(Chen & Guestrin, 2016, "XGBoost: A Scalable Tree Boosting System"). 이 노트북에서는 GradientBoost와 동일한 피처·타깃·트리 하이퍼파라미터로 실행해, "같은 원리(그레이디언트 부스팅)를 실무 최적화 구현체로 바꾸면 성능·속도가 어떻게 달라지는가"를 비교한다.

### F-2. 파라미터 설명(회귀·분류 공통)

| 파라미터 | 값 | 의미 |
|---|---|---|
| `n_estimators` | 100 | GradientBoost와 동일 — 순차적으로 쌓을 트리 개수 |
| `max_depth` | 3 | GradientBoost와 동일 — 트리 1개의 최대 깊이 |
| `learning_rate` | 0.05 | GradientBoost와 동일 — 각 트리의 기여 축소율 |
| `random_state` | 156 | 이 노트북 시드 관례 |
| `tree_method` | `'hist'` | XGBoost 전용 옵션 — 히스토그램 기반 분할 탐색으로 대용량 데이터 학습 속도를 크게 높인다. GradientBoost에는 없는 옵션이라 "공정 비교를 깨는 것 아닌가" 생각할 수 있으나, 이 옵션은 **분할 탐색의 근사 방식**이지 트리 개수·깊이·학습률 같은 "모델 용량" 파라미터가 아니다 — 기본값(`'exact'`)은 전체 데이터(148만 행)에서 실무적으로 쓸 수 없을 만큼 느려, `'hist'`가 XGBoost의 사실상 표준 설정이다. |
| `eval_metric`(분류만) | `'logloss'` | 최신 XGBoost 버전에서 평가지표를 명시적으로 고정해 경고를 방지하는 표준 관행 |

In [ ]:
# 기술적 의미: XGBoost 라이브러리에서 회귀용 클래스 XGBRegressor를 불러온다.
# 업무적 의미: 이 노트북 Part F의 회귀 담당 모델 — GradientBoost와 같은 목적(가격 예측)을 더 빠른 구현체로 다시 풀어본다.
from xgboost import XGBRegressor

# 기술적 의미: XGBRegressor를 GradientBoost(§Part A-2)와 동일한 n_estimators/max_depth/learning_rate/random_state로, 대용량 데이터 처리를 위한 tree_method='hist'를 추가해 생성한다. n_jobs=-1은 가용 CPU 코어를 모두 사용하라는 뜻이다.
# 업무적 의미: §F-2에서 설명한 대로 트리 개수·깊이·학습률을 GradientBoost와 동일하게 맞춰 "같은 조건" 비교가 되도록 했다 — X_train/X_test/y_train/y_test는 §Part A(위 코드 셀)에서 이미 만든 분할을 새로 만들지 않고 그대로 재사용한다.
xgbr = XGBRegressor(
    n_estimators=100, max_depth=3, learning_rate=0.05, random_state=156,
    tree_method='hist', n_jobs=-1,
)

# 기술적 의미: 학습 시작 시각을 기록한다.
# 업무적 의미: GradientBoost(Part A, 13,992.6초)와 동일한 방식으로 학습 소요시간을 실측해 §Part J 비교표에 반영하기 위함이다.
start = time.time()

# 기술적 의미: 학습 데이터로 XGBRegressor를 학습(fit)시킨다.
# 업무적 의미: "상품 정보로부터 가격을 배우는" 과정을 XGBoost 구현체로 다시 수행하는 단계다.
xgbr.fit(X_train, y_train)

# 기술적 의미: 학습 소요시간(초)을 계산한다.
# 업무적 의미: GradientBoost 대비 "같은 트리 개수·깊이인데 얼마나 더/덜 빠른가"를 실측 비교하기 위한 자료다.
fit_time_xgb = time.time() - start

# 기술적 의미: 검증 데이터(X_test)에 대한 예측값(log1p 공간)을 만든다.
# 업무적 의미: GradientBoost와 동일한 검증셋(X_test)에 대한 예측이므로, RMSLE·R² 값을 1:1로 직접 비교할 수 있다.
preds_xgb = xgbr.predict(X_test)

# 기술적 의미: evaluate_org_price()(§Part A에서 정의)를 그대로 재사용해 RMSLE를 계산한다.
# 업무적 의미: 회귀 평가 함수를 모델마다 새로 정의하지 않고 공유해, "평가 방식 차이"가 성능 차이에 섞여 들어가는 것을 막는다.
rmsle_xgb = evaluate_org_price(y_test, preds_xgb)

# 기술적 의미: 예측값을 expm1로 원래 가격 스케일로 복원한다.
# 업무적 의미: R² 계산을 GradientBoost와 동일한 스케일(달러)에서 수행하기 위한 준비 단계다.
preds_price_xgb = np.expm1(preds_xgb)

# 기술적 의미: 원래 가격 스케일에서 R²를 계산한다. 정답(y_test_price)은 §Part A에서 이미 계산해 둔 것을 재사용한다.
# 업무적 의미: GradientBoost의 r2(§A-3, 0.0906)와 나란히 비교할 수 있는 값이다.
r2_xgb = r2_score(y_test_price, preds_price_xgb)

# 기술적 의미: 학습 소요시간을 출력한다.
# 업무적 의미: GradientBoost 회귀(§A-3, 13,992.6초)와 직접 비교할 수 있는 실측치다.
print(f'[XGBoost 회귀] 학습 소요시간: {fit_time_xgb:.1f}초')

# 기술적 의미: RMSLE를 출력한다.
# 업무적 의미: GradientBoost 회귀(RMSLE 0.6535)와 직접 비교할 핵심 성과 지표다.
print(f'[XGBoost 회귀] RMSLE: {rmsle_xgb:.4f}')

# 기술적 의미: R²를 출력한다.
# 업무적 의미: GradientBoost 회귀(R² 0.0906)와 직접 비교할 보조 지표다.
print(f'[XGBoost 회귀] R2(결정계수): {r2_xgb:.4f}')

# 기술적 의미: XGBRegressor의 feature_importances_ 상위 10개를 §Part A-4(cell 9)에서 정의한 blocks/resolve_feature로 사람이 읽을 수 있는 이름으로 변환해 출력한다.
# 업무적 의미: GradientBoost(§A-4)와 같은 방식으로 "이 모델은 무엇을 보고 가격을 예측했는가"를 확인해, 알고리즘이 달라져도 상위 요인이 비슷한지(모델 신뢰도 교차검증) 다른지를 비교할 수 있게 한다.
top_idx_xgb = np.argsort(xgbr.feature_importances_)[::-1][:10]
print('[XGBoost 회귀] 가격 예측 기여 피처 top10:')
for i in top_idx_xgb:
    block, feat = resolve_feature(int(i))
    print(f'  중요도 {xgbr.feature_importances_[i]:.4f} | {block} | "{feat}"')


### F-3. XGBoost 회귀 결과 해석

> ⏸ 이 셀은 실행 대기 상태다 — 위 코드 셀을 실행한 뒤, 아래 항목을 실제 출력값으로 채운다.

- RMSLE·R²를 GradientBoost 회귀 참고치(RMSLE 0.6535, R² 0.0906, §A-3)와 비교해, XGBoost의 `tree_method='hist'`·정규화항이 실제로 성능 차이를 만드는지 확인할 것.
- 학습 소요시간을 GradientBoost(13,992.6초)와 비교해, 히스토그램 기반 분할이 대용량 데이터에서 실제로 얼마나 빠른지 정량적으로 확인할 것.
- top10 기여 피처가 GradientBoost의 top10(§A-3: `Other_Null` 브랜드, `lularoe`, `Shoes`, `Women's Handbags`, `box`, `Cell Phones & Smartphones`, `authentic` 등)과 얼마나 겹치는지 확인해, "알고리즘이 달라도 같은 업무 인사이트에 도달하는가"를 교차검증할 것.

### F-4. XGBoost 분류 — 배경과 파라미터

`XGBClassifier`는 `XGBRegressor`(§F-1~F-2)와 같은 트리 구조·학습 방식을 쓰되, 목적함수만 이진분류용 로그손실(`binary:logistic`, 자동 추론)로 바뀐다 — GradientBoost의 Part A→Part B 관계와 동일한 패턴이다. 하이퍼파라미터는 §F-2와 동일(`n_estimators=100, max_depth=3, learning_rate=0.05, random_state=156, tree_method='hist'`)하게 맞췄다.

In [ ]:
# 기술적 의미: XGBoost 라이브러리에서 분류용 클래스 XGBClassifier를 불러온다.
# 업무적 의미: 이 노트북 Part F의 분류 담당 모델 — GradientBoostingClassifier(Part B)와 같은 목적(배송비 부담 주체 분류)을 더 빠른 구현체로 다시 풀어본다.
from xgboost import XGBClassifier

# 기술적 의미: XGBClassifier를 회귀(§F-2/F-3)와 동일한 트리 하이퍼파라미터로 생성한다. eval_metric='logloss'는 §F-2에서 설명한 대로 최신 XGBoost 버전의 경고를 방지하기 위한 명시적 설정이다.
# 업무적 의미: GradientBoostingClassifier(Part B)와 동일 조건으로 맞춰, "그레이디언트 부스팅 구현체 차이"만 순수하게 비교할 수 있게 한다 — Xc_train/Xc_test/yc_train/yc_test는 §Part B에서 이미 만든 분류용 분할을 그대로 재사용한다.
xgbc = XGBClassifier(
    n_estimators=100, max_depth=3, learning_rate=0.05, random_state=156,
    tree_method='hist', n_jobs=-1, eval_metric='logloss',
)

# 기술적 의미: 분류 모델의 학습 시작 시각을 기록한다.
# 업무적 의미: GradientBoostingClassifier(Part B, 14,065.9초)와 동일한 방식으로 학습 소요시간을 실측·비교하기 위한 기준점이다.
start = time.time()

# 기술적 의미: 학습 데이터로 XGBClassifier를 학습(fit)시킨다.
# 업무적 의미: "상품 정보로부터 배송비 부담 주체를 배우는" 과정을 XGBoost 구현체로 다시 수행하는 단계다.
xgbc.fit(Xc_train, yc_train)

# 기술적 의미: 분류 모델의 학습 소요시간을 계산한다.
# 업무적 의미: GradientBoost 대비 XGBoost 분류의 속도 차이를 실측하기 위한 자료다.
fit_time_xgb_c = time.time() - start

# 기술적 의미: 학습된 모델로 검증 데이터에 대한 최종 예측 클래스(0 또는 1)를 만든다.
# 업무적 의미: "이 신규 상품은 무료배송일 것으로 예상되는가"를 최종적으로 한 가지 값으로 판정하는 결과다.
pred_class_xgb = xgbc.predict(Xc_test)

# 기술적 의미: predict_proba()로 각 클래스일 확률을 구한 뒤, 클래스 1(판매자부담)일 확률만 골라 pred_proba_xgb에 저장한다.
# 업무적 의미: GradientBoost(Part B)와 동일하게, 이분법적 판정 대신 확률값을 남겨 이후 임계값 조정 등 업무 활용이 가능해진다.
pred_proba_xgb = xgbc.predict_proba(Xc_test)[:, 1]

# 기술적 의미: 예측 클래스와 실제 클래스를 비교해 정확도를 계산한다.
# 업무적 의미: GradientBoost 분류(Accuracy 0.6707, §B-3)와 직접 비교할 지표다.
acc_xgb = accuracy_score(yc_test, pred_class_xgb)

# 기술적 의미: 예측 확률과 실제 클래스를 비교해 ROC-AUC를 계산한다.
# 업무적 의미: GradientBoost 분류(AUC 0.7123, §B-3)와 직접 비교할 지표다.
auc_xgb = roc_auc_score(yc_test, pred_proba_xgb)

# 기술적 의미: recall_score(재현율)를 average='macro'로 계산한다.
# 업무적 의미: 두 클래스(구매자부담/판매자부담)를 균형 있게 잘 맞히는지, GradientBoost(Recall macro 0.6533, §B-3)와 비교할 지표다.
recall_xgb = recall_score(yc_test, pred_class_xgb, average='macro')

# 기술적 의미: f1_score(정밀도·재현율의 조화평균)를 average='macro'로 계산한다.
# 업무적 의미: GradientBoost(F1 macro 0.6517, §B-3)와 비교할 종합 지표다.
f1_xgb = f1_score(yc_test, pred_class_xgb, average='macro')

# 기술적 의미: precision_score(정밀도)도 참고용으로 macro 평균을 계산해 둔다.
# 업무적 의미: recall/f1과 함께 해석해야 트레이드오프를 정확히 판단할 수 있다(§Part B와 동일한 이유).
precision_xgb = precision_score(yc_test, pred_class_xgb, average='macro')

# 기술적 의미: 학습 소요시간을 출력한다.
# 업무적 의미: GradientBoost 분류(14,065.9초)와 나란히 비교할 수 있는 실측 자료를 남긴다.
print(f'[XGBoost 분류] 학습 소요시간: {fit_time_xgb_c:.1f}초')

# 기술적 의미: 핵심 지표 4종 출력 전 안내 문구를 표시한다.
# 업무적 의미: GradientBoost(Part B)와 동일한 형식으로 "업무에서 바로 확인해야 할 4대 지표"를 구분해 보여준다.
print('--- [XGBoost 분류] 핵심 지표 요약(accuracy / roc_auc / recall / f1_score) ---')

# 기술적 의미: 정확도를 소수점 4자리로 출력한다.
print(f'Accuracy(정확도): {acc_xgb:.4f}')

# 기술적 의미: ROC-AUC를 소수점 4자리로 출력한다.
print(f'ROC-AUC: {auc_xgb:.4f}')

# 기술적 의미: macro 평균 재현율을 소수점 4자리로 출력한다.
print(f'Recall(재현율, macro 평균): {recall_xgb:.4f}')

# 기술적 의미: macro 평균 F1-score를 소수점 4자리로 출력한다.
print(f'F1-score(macro 평균): {f1_xgb:.4f}')

# 기술적 의미: 참고용으로 macro 평균 정밀도도 함께 출력한다.
print(f'Precision(정밀도, macro 평균, 참고): {precision_xgb:.4f}')

# 기술적 의미: 출력 사이에 빈 줄을 넣어 가독성을 높인다.
print()

# 기술적 의미: 클래스별 precision·recall·f1-score·support를 표 형태로 출력한다.
# 업무적 의미: macro 평균만으로는 안 보이는 "어느 클래스가 더 취약한가"를 GradientBoost(Part B)와 동일한 형식으로 확인한다.
print(classification_report(yc_test, pred_class_xgb, target_names=['구매자부담(0)', '판매자부담(1)']))

# 기술적 의미: 혼동행렬 출력 전 안내 문구를 표시한다.
print('혼동행렬(행=실제, 열=예측):')

# 기술적 의미: 실제 클래스와 예측 클래스로 2x2 혼동행렬을 만들어 출력한다.
# 업무적 의미: GradientBoost(§B-3, 오분류 1,311건/4,000건 — 20,000행 표본 당시 값)와 오분류 패턴(어느 방향 오류가 많은지)을 비교할 수 있는 자료다.
print(confusion_matrix(yc_test, pred_class_xgb))

# 기술적 의미: XGBClassifier의 feature_importances_ 상위 10개를 사람이 읽을 수 있는 이름으로 변환해 출력한다.
# 업무적 의미: "배송비 부담 주체를 가르는 결정적 요인"이 회귀(가격)와 다른지 같은지 비교할 수 있는 해석 자료다.
top_idx_xgb_c = np.argsort(xgbc.feature_importances_)[::-1][:10]
print('[XGBoost 분류] 배송비 부담 예측 기여 피처 top10:')
for i in top_idx_xgb_c:
    block, feat = resolve_feature(int(i))
    print(f'  중요도 {xgbc.feature_importances_[i]:.4f} | {block} | "{feat}"')


### F-5. XGBoost 분류 결과 해석

> ⏸ 이 셀은 실행 대기 상태다 — 위 코드 셀을 실행한 뒤, 아래 항목을 실제 출력값으로 채운다.

- Accuracy/ROC-AUC/Recall/F1을 GradientBoost 분류 참고치(Accuracy 0.6707, AUC 0.7123, Recall macro 0.6533, F1 macro 0.6517, §B-3)와 비교할 것.
- 클래스별 recall(구매자부담 vs 판매자부담) 격차가 GradientBoost와 비슷한 패턴(다수 클래스 쪽으로 치우침)인지, 아니면 XGBoost의 정규화항이 이 불균형을 완화하는지 확인할 것.
- 학습 소요시간을 GradientBoost 분류(14,065.9초)와 비교할 것.

## Part G. LightGBM — 가격(회귀) / 배송비(분류)

### G-1. 배경과 사용 이유(요약)

LightGBM은 XGBoost와 같은 히스토그램 기반 그레이디언트 부스팅이지만, 트리를 **리프 단위(leaf-wise)**로 키운다는 점이 가장 큰 차이다(Microsoft, Ke et al., 2017, "LightGBM: A Highly Efficient Gradient Boosting Decision Tree") — sklearn GradientBoost·XGBoost 기본 방식인 **레벨 단위(level-wise, 깊이 우선)** 성장과 달리, "가장 손실을 많이 줄이는 리프"부터 먼저 분할해 같은 트리 개수·깊이 제한에서도 더 빠르게 손실을 줄이는 경향이 있다. 원본 노트북(`10 캐글 mercari price_정제후.ipynb`)에서 이미 LightGBM으로 실측한 바 있으나(RMSLE 0.4564, 전체 148만 행 기준, `shipping` 피처 **포함** 조건) 조건이 이 노트북과 다르므로, 이 노트북에서는 **GradientBoost와 동일한 조건(피처셋에서 `shipping` 제외 포함, §2-4)**으로 다시 실행해 공정하게 비교한다.

### G-2. 파라미터 설명(회귀·분류 공통)

| 파라미터 | 값 | 의미 |
|---|---|---|
| `n_estimators` | 100 | GradientBoost와 동일 |
| `max_depth` | 3 | GradientBoost와 동일 |
| `learning_rate` | 0.05 | GradientBoost와 동일 |
| `random_state` | 156 | 이 노트북 시드 관례 |
| `num_leaves` | 8(=2³) | LightGBM 전용 — 리프 단위 성장에서는 `num_leaves`(기본값 31)가 실질적인 복잡도 상한이다. `max_depth=3`만 주면 리프 단위 성장 특성상 결과적으로 GradientBoost의 "얕은 트리"보다 복잡한 트리가 만들어질 수 있어, 2³=8로 명시적으로 함께 제한해 실질적인 모델 용량을 맞췄다. |
| `verbose` | -1 | 학습 중 반복마다 출력되는 LightGBM 내부 로그를 꺼서 노트북 출력이 과도하게 길어지는 것을 방지한다 |

In [ ]:
# 기술적 의미: lightgbm 라이브러리에서 회귀용 클래스 LGBMRegressor를 불러온다.
# 업무적 의미: 이 노트북 Part G의 회귀 담당 모델 — 리프 단위 성장 방식의 그레이디언트 부스팅으로 같은 가격 예측 문제를 다시 풀어본다.
from lightgbm import LGBMRegressor

# 기술적 의미: LGBMRegressor를 GradientBoost와 동일한 n_estimators/max_depth/learning_rate/random_state에, §G-2에서 설명한 num_leaves=8과 verbose=-1을 추가해 생성한다.
# 업무적 의미: 리프 단위 성장이라는 알고리즘 차이를 제외한 나머지 조건(트리 개수·깊이 상한·학습률·데이터·분할)을 GradientBoost와 동일하게 맞춘 것 — X_train/X_test/y_train/y_test는 §Part A에서 만든 분할을 그대로 재사용한다.
lgbmr = LGBMRegressor(
    n_estimators=100, max_depth=3, learning_rate=0.05, random_state=156,
    num_leaves=8, verbose=-1,
)

# 기술적 의미: 학습 시작 시각을 기록한다.
start = time.time()

# 기술적 의미: 학습 데이터로 LGBMRegressor를 학습(fit)시킨다.
# 업무적 의미: 리프 단위 성장 방식으로 "상품 정보로부터 가격을 배우는" 과정을 다시 수행하는 단계다.
lgbmr.fit(X_train, y_train)

# 기술적 의미: 학습 소요시간(초)을 계산한다.
# 업무적 의미: GradientBoost(13,992.6초)·XGBoost와 나란히 비교할 실측 자료다.
fit_time_lgbm = time.time() - start

# 기술적 의미: 검증 데이터에 대한 예측값(log1p 공간)을 만든다.
preds_lgbm = lgbmr.predict(X_test)

# 기술적 의미: evaluate_org_price()를 재사용해 RMSLE를 계산한다.
# 업무적 의미: GradientBoost(RMSLE 0.6535)·XGBoost와 동일 기준으로 비교할 핵심 성과 지표다.
rmsle_lgbm = evaluate_org_price(y_test, preds_lgbm)

# 기술적 의미: 예측값을 expm1로 원래 가격 스케일로 복원한다.
preds_price_lgbm = np.expm1(preds_lgbm)

# 기술적 의미: 원래 가격 스케일에서 R²를 계산한다.
# 업무적 의미: GradientBoost(R² 0.0906)와 비교할 보조 지표다.
r2_lgbm = r2_score(y_test_price, preds_price_lgbm)

# 기술적 의미: 학습 소요시간을 출력한다.
print(f'[LightGBM 회귀] 학습 소요시간: {fit_time_lgbm:.1f}초')

# 기술적 의미: RMSLE를 출력한다.
print(f'[LightGBM 회귀] RMSLE: {rmsle_lgbm:.4f}')

# 기술적 의미: R²를 출력한다.
print(f'[LightGBM 회귀] R2(결정계수): {r2_lgbm:.4f}')

# 기술적 의미: LGBMRegressor의 feature_importances_(기본값: 트리에서 해당 피처로 분할한 횟수) 상위 10개를 사람이 읽을 수 있는 이름으로 변환해 출력한다.
# 업무적 의미: GradientBoost·XGBoost와 상위 요인이 얼마나 겹치는지 비교해, 알고리즘이 달라도 같은 업무 인사이트(예: 브랜드 미기재, 신발/가방 카테고리)에 도달하는지 교차검증한다.
top_idx_lgbm = np.argsort(lgbmr.feature_importances_)[::-1][:10]
print('[LightGBM 회귀] 가격 예측 기여 피처 top10:')
for i in top_idx_lgbm:
    block, feat = resolve_feature(int(i))
    print(f'  중요도 {lgbmr.feature_importances_[i]:.4f} | {block} | "{feat}"')


### G-3. LightGBM 회귀 결과 해석

> ⏸ 이 셀은 실행 대기 상태다 — 위 코드 셀을 실행한 뒤, 아래 항목을 실제 출력값으로 채운다.

- RMSLE·R²를 GradientBoost(RMSLE 0.6535, R² 0.0906)·XGBoost(§F-3)와 함께 3자 비교할 것.
- 학습 소요시간을 GradientBoost(13,992.6초)와 비교해, 리프 단위 성장 + 히스토그램 분할이 실제로 얼마나 빠른지 확인할 것 — 원본 노트북(10번)에서도 LightGBM이 실무적으로 널리 쓰이는 이유 중 하나가 속도였다.
- top10 기여 피처가 GradientBoost·XGBoost의 top10과 얼마나 겹치는지 확인할 것.

### G-4. LightGBM 분류 — 배경과 파라미터

`LGBMClassifier`는 `LGBMRegressor`(§G-1~G-2)와 같은 리프 단위 성장 방식을 쓰되, 목적함수만 이진분류용 로그손실로 바뀐다. 하이퍼파라미터는 §G-2와 동일(`n_estimators=100, max_depth=3, learning_rate=0.05, random_state=156, num_leaves=8, verbose=-1`)하게 맞췄다.

In [ ]:
# 기술적 의미: lightgbm 라이브러리에서 분류용 클래스 LGBMClassifier를 불러온다.
# 업무적 의미: 이 노트북 Part G의 분류 담당 모델 — GradientBoostingClassifier(Part B)·XGBClassifier(Part F)와 같은 목적(배송비 부담 주체 분류)을 리프 단위 성장 구현체로 다시 풀어본다.
from lightgbm import LGBMClassifier

# 기술적 의미: LGBMClassifier를 회귀(§G-2/G-3)와 동일한 하이퍼파라미터로 생성한다.
# 업무적 의미: Xc_train/Xc_test/yc_train/yc_test는 §Part B에서 만든 분류용 분할을 그대로 재사용해, 다른 4개 모델과 동일 조건에서 비교되도록 한다.
lgbmc = LGBMClassifier(
    n_estimators=100, max_depth=3, learning_rate=0.05, random_state=156,
    num_leaves=8, verbose=-1,
)

# 기술적 의미: 분류 모델의 학습 시작 시각을 기록한다.
start = time.time()

# 기술적 의미: 학습 데이터로 LGBMClassifier를 학습(fit)시킨다.
lgbmc.fit(Xc_train, yc_train)

# 기술적 의미: 분류 모델의 학습 소요시간을 계산한다.
# 업무적 의미: GradientBoost(14,065.9초)·XGBoost와 나란히 비교할 실측 자료다.
fit_time_lgbm_c = time.time() - start

# 기술적 의미: 검증 데이터에 대한 최종 예측 클래스(0 또는 1)를 만든다.
pred_class_lgbm = lgbmc.predict(Xc_test)

# 기술적 의미: 클래스 1(판매자부담)일 확률을 구한다.
pred_proba_lgbm = lgbmc.predict_proba(Xc_test)[:, 1]

# 기술적 의미: 정확도를 계산한다.
acc_lgbm = accuracy_score(yc_test, pred_class_lgbm)

# 기술적 의미: ROC-AUC를 계산한다.
auc_lgbm = roc_auc_score(yc_test, pred_proba_lgbm)

# 기술적 의미: macro 평균 재현율을 계산한다.
recall_lgbm = recall_score(yc_test, pred_class_lgbm, average='macro')

# 기술적 의미: macro 평균 F1-score를 계산한다.
f1_lgbm = f1_score(yc_test, pred_class_lgbm, average='macro')

# 기술적 의미: macro 평균 정밀도를 참고용으로 계산한다.
precision_lgbm = precision_score(yc_test, pred_class_lgbm, average='macro')

# 기술적 의미: 학습 소요시간을 출력한다.
print(f'[LightGBM 분류] 학습 소요시간: {fit_time_lgbm_c:.1f}초')

# 기술적 의미: 핵심 지표 4종 출력 전 안내 문구를 표시한다.
print('--- [LightGBM 분류] 핵심 지표 요약(accuracy / roc_auc / recall / f1_score) ---')

# 기술적 의미: 정확도를 출력한다.
print(f'Accuracy(정확도): {acc_lgbm:.4f}')

# 기술적 의미: ROC-AUC를 출력한다.
print(f'ROC-AUC: {auc_lgbm:.4f}')

# 기술적 의미: macro 평균 재현율을 출력한다.
print(f'Recall(재현율, macro 평균): {recall_lgbm:.4f}')

# 기술적 의미: macro 평균 F1-score를 출력한다.
print(f'F1-score(macro 평균): {f1_lgbm:.4f}')

# 기술적 의미: macro 평균 정밀도를 참고용으로 출력한다.
print(f'Precision(정밀도, macro 평균, 참고): {precision_lgbm:.4f}')

# 기술적 의미: 출력 사이에 빈 줄을 넣는다.
print()

# 기술적 의미: 클래스별 상세 성적표를 출력한다.
print(classification_report(yc_test, pred_class_lgbm, target_names=['구매자부담(0)', '판매자부담(1)']))

# 기술적 의미: 혼동행렬 출력 전 안내 문구를 표시한다.
print('혼동행렬(행=실제, 열=예측):')

# 기술적 의미: 2x2 혼동행렬을 출력한다.
print(confusion_matrix(yc_test, pred_class_lgbm))

# 기술적 의미: LGBMClassifier의 feature_importances_ 상위 10개를 사람이 읽을 수 있는 이름으로 변환해 출력한다.
top_idx_lgbm_c = np.argsort(lgbmc.feature_importances_)[::-1][:10]
print('[LightGBM 분류] 배송비 부담 예측 기여 피처 top10:')
for i in top_idx_lgbm_c:
    block, feat = resolve_feature(int(i))
    print(f'  중요도 {lgbmc.feature_importances_[i]:.4f} | {block} | "{feat}"')


### G-5. LightGBM 분류 결과 해석

> ⏸ 이 셀은 실행 대기 상태다 — 위 코드 셀을 실행한 뒤, 아래 항목을 실제 출력값으로 채운다.

- Accuracy/ROC-AUC/Recall/F1을 GradientBoost(§B-3)·XGBoost(§F-5)와 함께 3자 비교할 것.
- 학습 소요시간이 세 그레이디언트 부스팅 모델 중 가장 짧은지 확인할 것(리프 단위 성장 + 히스토그램 분할의 실무적 장점).

## Part H. LinearRegression — 가격(회귀) 기준점(baseline)

### H-1. 배경과 사용 이유(요약)

`LinearRegression`은 모든 피처의 가중합(선형결합)만으로 가격(log1p)을 직접 추정하는 가장 단순한 회귀 모델이다. 트리 기반 앙상블 3종(GradientBoost/XGBoost/LightGBM)이 "복잡한 비선형 모델이 실제로 얼마나 더 나은가"를 판단하려면, 반드시 이런 단순한 **기준점(baseline)**과 비교해야 한다 — 만약 LinearRegression과 성능 차이가 크지 않다면, 트리 앙상블의 학습·서빙 비용(§Part C, 61초 vs 트리모델 수 시간)을 감수할 실익이 낮다는 뜻이 된다.

### H-2. 파라미터 설명

| 파라미터 | 값 | 의미 |
|---|---|---|
| (해당 없음) | 기본값 | `LinearRegression`은 트리 개수·깊이 같은 "모델 용량" 파라미터가 없다 — 데이터가 주어지면 최소제곱 해가 수학적으로 유일하게 정해지는 모델이라, `random_state`조차 필요 없다(실행마다 결과가 항상 동일). GradientBoost와 "맞출 하이퍼파라미터 자체가 없다"는 것이 이 모델의 특징이다. |

In [ ]:
# 기술적 의미: sklearn에서 가장 단순한 선형회귀 모델 LinearRegression을 불러온다.
# 업무적 의미: 트리 기반 앙상블(GradientBoost/XGBoost/LightGBM)과 대비되는 "선형 모델 기준점(baseline)"을 만들기 위함이다 — §H-1에서 설명한 대로, 복잡한 모델이 실제로 단순한 선형모델보다 얼마나 더 나은지 판단하려면 반드시 기준점이 있어야 한다.
from sklearn.linear_model import LinearRegression

# 기술적 의미: LinearRegression을 기본 설정으로 생성한다.
# 업무적 의미: §H-2에서 설명한 대로 이 모델은 맞출 하이퍼파라미터가 없다.
linreg = LinearRegression()

# 기술적 의미: 학습 시작 시각을 기록한다.
# 업무적 의미: 트리 앙상블 3종과 동일한 방식으로 학습 소요시간을 실측해 "속도 대 성능" 트레이드오프를 비교하기 위함이다.
start = time.time()

# 기술적 의미: X_train(sparse 피처 행렬)과 y_train으로 LinearRegression을 학습시킨다. sklearn은 sparse 입력을 받으면 내부적으로 scipy의 반복적 최소제곱 해법(lsqr)을 사용한다.
# 업무적 의미: 트리 기반 모델과 전혀 다른 원리(모든 피처의 가중합으로 가격을 직접 추정)로 같은 문제를 풀어보는 단계다 — X_train/y_train은 §Part A에서 만든 분할을 그대로 재사용한다.
linreg.fit(X_train, y_train)

# 기술적 의미: 학습 소요시간을 계산한다.
# 업무적 의미: 트리 앙상블(수십 분~수 시간) 대비 선형모델이 얼마나 빠른지 실측으로 보여주는 자료 — §Part J 비교표의 핵심 근거 중 하나다.
fit_time_linreg = time.time() - start

# 기술적 의미: 검증 데이터에 대한 예측값(log1p 공간)을 만든다.
# 업무적 의미: 다른 모델과 동일한 검증셋(X_test)에 대한 예측이므로 RMSLE·R²를 직접 비교할 수 있다.
preds_linreg = linreg.predict(X_test)

# 기술적 의미: 선형회귀는 예측값이 음수가 될 수 있어, expm1 복원 후 RMSLE 계산 시 log1p(음수)가 NaN을 만들 위험이 있다 — 가격은 0 미만일 수 없으므로 예측값을 0 이상으로 잘라낸다(clip).
# 업무적 의미: "모델이 -3달러짜리 상품을 예측했다"는 것은 업무적으로 말이 안 되는 값이다 — 실제 서비스라면 이런 값을 그대로 노출하면 안 되므로, 평가 단계에서부터 현실적인 하한(0원)을 반영하는 것이 올바른 처리다.
preds_price_linreg = np.clip(np.expm1(preds_linreg), a_min=0, a_max=None)

# 기술적 의미: RMSLE를 원래 가격 스케일에서 직접 계산한다(§Part A에서 정의한 rmsle() 재사용, clip으로 이미 원 스케일 복원을 마쳤으므로 evaluate_org_price() 대신 rmsle()를 바로 호출).
# 업무적 의미: GradientBoost(RMSLE 0.6535)와 비교할 핵심 성과 지표다.
rmsle_linreg = rmsle(y_test_price, preds_price_linreg)

# 기술적 의미: R²를 계산한다.
# 업무적 의미: 선형모델의 설명력을 트리 앙상블과 비교할 보조 지표다.
r2_linreg = r2_score(y_test_price, preds_price_linreg)

# 기술적 의미: 학습 소요시간을 출력한다.
print(f'[LinearRegression] 학습 소요시간: {fit_time_linreg:.1f}초')

# 기술적 의미: RMSLE를 출력한다.
print(f'[LinearRegression] RMSLE: {rmsle_linreg:.4f}')

# 기술적 의미: R²를 출력한다.
print(f'[LinearRegression] R2(결정계수): {r2_linreg:.4f}')

# 기술적 의미: 회귀계수(coef_) 절댓값이 큰 상위 10개 피처를 blocks/resolve_feature로 사람이 읽을 수 있는 이름으로 변환해 출력한다.
# 업무적 의미: 트리 모델의 "중요도"와 달리 선형모델의 "계수"는 "그 피처가 1 증가할 때 log(가격)이 얼마나, 어느 방향(+/-)으로 변하는가"까지 알려준다 — 트리 모델과는 다른 해석 관점을 제공한다.
top_idx_linreg = np.argsort(np.abs(linreg.coef_))[::-1][:10]
print('[LinearRegression] 가격에 가장 큰 영향을 준 피처 top10(|계수| 기준):')
for i in top_idx_linreg:
    block, feat = resolve_feature(int(i))
    print(f'  계수 {linreg.coef_[i]:+.4f} | {block} | "{feat}"')


### H-3. LinearRegression 결과 해석

> ⏸ 이 셀은 실행 대기 상태다 — 위 코드 셀을 실행한 뒤, 아래 항목을 실제 출력값으로 채운다.

- RMSLE·R²가 트리 앙상블 3종(GradientBoost RMSLE 0.6535 등)보다 뚜렷이 나쁘다면, "브랜드×카테고리×텍스트가 복합적으로 상호작용해 가격을 결정한다"는 이 데이터의 비선형적 특성을 트리 모델이 실제로 포착하고 있다는 근거가 된다.
- 반대로 성능 차이가 작다면, 이 피처셋에서는 단순 선형결합만으로도 상당한 설명력을 얻을 수 있다는 뜻이므로 서빙 비용(학습·추론 속도) 대비 트리 모델의 실익을 재검토할 근거가 된다.
- 학습 소요시간을 트리 앙상블 3종과 비교해 "속도 대 성능" 트레이드오프를 §Part J에서 종합 판단할 것.

## Part I. LogisticRegression — 배송비(분류) 기준점(baseline)

### I-1. 배경과 사용 이유(요약)

`LogisticRegression`은 모든 피처의 가중합에 시그모이드 함수를 적용해 "판매자부담(1)일 확률"을 직접 추정하는 가장 단순한 이진분류 모델이다. Part H(LinearRegression)가 회귀의 기준점 역할을 하듯, 이 모델은 트리 기반 분류기 3종(GradientBoost/XGBoost/LightGBM Classifier)의 **기준점(baseline)** 역할을 한다.

### I-2. 파라미터 설명

| 파라미터 | 값 | 의미 |
|---|---|---|
| `solver` | `'liblinear'` | 대용량 희소(sparse) 이진분류에서 실무적으로 가장 널리 쓰이는 표준 solver — 좌표하강 기반으로 L2 정규화·이진분류·sparse 입력에 안정적으로 동작한다. |
| `max_iter` | 200 | 기본값(100)으로는 이 정도 규모(161,568차원, 118만+ 학습 표본)의 데이터에서 수렴이 부족할 수 있어 여유를 두었다. |
| `random_state` | 156 | 이 노트북 시드 관례(liblinear 내부 셔플링의 재현성 보장) |

In [ ]:
# 기술적 의미: sklearn에서 로지스틱회귀 모델 LogisticRegression을 불러온다.
# 업무적 의미: §I-1에서 설명한 대로, 트리 기반 분류기 3종과 대비되는 "선형 모델 기준점(baseline)"을 만들기 위함이다.
from sklearn.linear_model import LogisticRegression

# 기술적 의미: LogisticRegression을 solver='liblinear'(§I-2 설명), max_iter=200, random_state=156으로 생성한다.
# 업무적 의미: 대용량 희소 이진분류에 적합한 표준 설정이며, GradientBoostingClassifier처럼 맞출 트리 파라미터는 없지만 "왜 이 solver·반복횟수를 골랐는가"를 명시해 자의적 선택이 아님을 밝힌다.
logreg = LogisticRegression(solver='liblinear', max_iter=200, random_state=156)

# 기술적 의미: 학습 시작 시각을 기록한다.
start = time.time()

# 기술적 의미: Xc_train/yc_train(§Part B에서 만든 분류용 학습 분할, stratify 적용됨)으로 LogisticRegression을 학습시킨다.
# 업무적 의미: GradientBoostingClassifier(Part B)와 동일한 학습·검증 분할을 사용해 직접 비교 가능한 결과를 만든다.
logreg.fit(Xc_train, yc_train)

# 기술적 의미: 학습 소요시간을 계산한다.
# 업무적 의미: 트리 기반 분류기 3종 대비 선형모델의 속도를 실측하는 자료다.
fit_time_logreg = time.time() - start

# 기술적 의미: 검증 데이터에 대한 최종 예측 클래스(0/1)를 만든다.
pred_class_logreg = logreg.predict(Xc_test)

# 기술적 의미: 클래스 1(판매자부담)일 확률을 구한다 — ROC-AUC 계산에 필요하다.
pred_proba_logreg = logreg.predict_proba(Xc_test)[:, 1]

# 기술적 의미: 정확도를 계산한다.
acc_logreg = accuracy_score(yc_test, pred_class_logreg)

# 기술적 의미: ROC-AUC를 계산한다.
auc_logreg = roc_auc_score(yc_test, pred_proba_logreg)

# 기술적 의미: macro 평균 재현율을 계산한다.
recall_logreg = recall_score(yc_test, pred_class_logreg, average='macro')

# 기술적 의미: macro 평균 F1-score를 계산한다.
f1_logreg = f1_score(yc_test, pred_class_logreg, average='macro')

# 기술적 의미: macro 평균 정밀도를 참고용으로 계산한다.
precision_logreg = precision_score(yc_test, pred_class_logreg, average='macro')

# 기술적 의미: 학습 소요시간을 출력한다.
print(f'[LogisticRegression] 학습 소요시간: {fit_time_logreg:.1f}초')

# 기술적 의미: 핵심 지표 4종 출력 전 안내 문구를 표시한다.
print('--- [LogisticRegression] 핵심 지표 요약(accuracy / roc_auc / recall / f1_score) ---')

# 기술적 의미: 정확도를 출력한다.
print(f'Accuracy(정확도): {acc_logreg:.4f}')

# 기술적 의미: ROC-AUC를 출력한다.
print(f'ROC-AUC: {auc_logreg:.4f}')

# 기술적 의미: macro 평균 재현율을 출력한다.
print(f'Recall(재현율, macro 평균): {recall_logreg:.4f}')

# 기술적 의미: macro 평균 F1-score를 출력한다.
print(f'F1-score(macro 평균): {f1_logreg:.4f}')

# 기술적 의미: macro 평균 정밀도를 참고용으로 출력한다.
print(f'Precision(정밀도, macro 평균, 참고): {precision_logreg:.4f}')

# 기술적 의미: 출력 사이에 빈 줄을 넣는다.
print()

# 기술적 의미: 클래스별 상세 성적표를 출력한다.
print(classification_report(yc_test, pred_class_logreg, target_names=['구매자부담(0)', '판매자부담(1)']))

# 기술적 의미: 혼동행렬 출력 전 안내 문구를 표시한다.
print('혼동행렬(행=실제, 열=예측):')

# 기술적 의미: 2x2 혼동행렬을 출력한다.
print(confusion_matrix(yc_test, pred_class_logreg))

# 기술적 의미: 회귀계수(coef_[0], 이진분류이므로 클래스 1에 대한 계수 1세트만 존재) 절댓값 상위 10개를 출력한다.
# 업무적 의미: "어떤 상품 정보가 판매자부담(무료배송) 여부를 가르는 결정적 신호인가"를 계수의 방향(+/-)까지 함께 보여준다.
top_idx_logreg = np.argsort(np.abs(logreg.coef_[0]))[::-1][:10]
print('[LogisticRegression] 배송비 부담 예측에 가장 큰 영향을 준 피처 top10(|계수| 기준):')
for i in top_idx_logreg:
    block, feat = resolve_feature(int(i))
    print(f'  계수 {logreg.coef_[0][i]:+.4f} | {block} | "{feat}"')


### I-3. LogisticRegression 결과 해석

> ⏸ 이 셀은 실행 대기 상태다 — 위 코드 셀을 실행한 뒤, 아래 항목을 실제 출력값으로 채운다.

- Accuracy/ROC-AUC를 트리 기반 분류기 3종(GradientBoost AUC 0.7123 등)과 비교해, 선형 결정 경계만으로 배송비 부담 주체를 얼마나 잘 가르는지 확인할 것.
- 계수 부호(+/-)가 실제 업무 상식(예: 저가 상품일수록 무료배송 경향)과 일치하는지 확인할 것 — 트리 모델의 "중요도"에는 없는, 선형모델만의 해석 장점이다.

## Part J. 5개 모델 종합 비교

```mermaid
flowchart LR
    A["회귀 4종<br/>GradientBoost/XGBoost/LightGBM/LinearRegression"] --> B["RMSLE·R²·학습시간<br/>비교표(오름차순: RMSLE)"]
    C["분류 4종<br/>GradientBoost/XGBoost/LightGBM/LogisticRegression"] --> D["Accuracy·AUC·Recall·F1·학습시간<br/>비교표(내림차순: AUC)"]
    B --> E["Part J-2 해석<br/>'성능 vs 속도' 트레이드오프 판단"]
    D --> E
```

### J-1. 비교표 생성

In [ ]:
# 기술적 의미: 5개 모델 중 회귀(price) 결과가 있는 4개 모델(GradientBoost/XGBoost/LightGBM/LinearRegression)의 RMSLE·R²·학습시간을 한 표로 모은다.
# 업무적 의미: Part A/F/G/H에 흩어져 있던 회귀 성능을 한눈에 비교할 수 있는 최종 비교표를 만드는 단계 — 의사결정권자가 "어떤 모델을 쓸 것인가"를 이 표 하나로 판단할 수 있게 한다.
regression_comparison = pd.DataFrame([
    {'모델': 'GradientBoost', 'RMSLE': rmsle_value, 'R2': r2, '학습시간(초)': fit_time},
    {'모델': 'XGBoost', 'RMSLE': rmsle_xgb, 'R2': r2_xgb, '학습시간(초)': fit_time_xgb},
    {'모델': 'LightGBM', 'RMSLE': rmsle_lgbm, 'R2': r2_lgbm, '학습시간(초)': fit_time_lgbm},
    {'모델': 'LinearRegression', 'RMSLE': rmsle_linreg, 'R2': r2_linreg, '학습시간(초)': fit_time_linreg},
]).sort_values('RMSLE').reset_index(drop=True)

# 기술적 의미: 회귀 비교표를 안내 문구와 함께 출력한다.
# 업무적 의미: RMSLE가 작을수록(0에 가까울수록) 좋은 모델이므로, 오름차순 정렬로 "가장 좋은 모델이 맨 위"에 오도록 했다.
print('=== 회귀(price) 모델 비교 — RMSLE 오름차순 ===')
print(regression_comparison.to_string(index=False))
print()

# 기술적 의미: 5개 모델 중 분류(shipping) 결과가 있는 4개 모델(GradientBoost/XGBoost/LightGBM/LogisticRegression)의 Accuracy·AUC·Recall·F1·학습시간을 한 표로 모은다.
# 업무적 의미: 회귀와 마찬가지로, 분류 성능도 한 표로 모아 바로 비교할 수 있게 한다.
classification_comparison = pd.DataFrame([
    {'모델': 'GradientBoost', 'Accuracy': acc, 'AUC': auc, 'Recall(macro)': recall_macro, 'F1(macro)': f1_macro, '학습시간(초)': fit_time_c},
    {'모델': 'XGBoost', 'Accuracy': acc_xgb, 'AUC': auc_xgb, 'Recall(macro)': recall_xgb, 'F1(macro)': f1_xgb, '학습시간(초)': fit_time_xgb_c},
    {'모델': 'LightGBM', 'Accuracy': acc_lgbm, 'AUC': auc_lgbm, 'Recall(macro)': recall_lgbm, 'F1(macro)': f1_lgbm, '학습시간(초)': fit_time_lgbm_c},
    {'모델': 'LogisticRegression', 'Accuracy': acc_logreg, 'AUC': auc_logreg, 'Recall(macro)': recall_logreg, 'F1(macro)': f1_logreg, '학습시간(초)': fit_time_logreg},
]).sort_values('AUC', ascending=False).reset_index(drop=True)

# 기술적 의미: 분류 비교표를 안내 문구와 함께 출력한다.
# 업무적 의미: AUC가 클수록 판별력이 좋은 모델이므로, 내림차순 정렬로 "가장 좋은 모델이 맨 위"에 오도록 했다.
print('=== 분류(shipping) 모델 비교 — AUC 내림차순 ===')
print(classification_comparison.to_string(index=False))


### J-2. 종합 해석

> ⏸ 이 셀은 실행 대기 상태다 — 위 비교표를 실제로 생성한 뒤, 아래 항목을 채운다.

- **성능**: RMSLE(회귀)·AUC(분류) 기준 1위 모델이 어느 것인지, 그리고 그 차이가 업무적으로 유의미한 수준인지(예: RMSLE 0.01 차이는 사실상 무시 가능한 수준일 수 있다) 판단할 것.
- **속도**: 학습시간이 성능 차이를 정당화할 만큼 벌어지는지 확인할 것 — GradientBoost는 회귀·분류 각각 약 3.9시간(13,992.6초/14,065.9초)이 걸렸으므로, XGBoost·LightGBM이 비슷한 성능을 훨씬 짧은 시간에 낸다면 실무에서는 그쪽을 선택하는 것이 합리적이다.
- **선형 기준점과의 격차**: LinearRegression·LogisticRegression과 트리 앙상블의 성능 격차가 크다면 "이 문제는 비선형 상호작용이 중요하다"는 근거가 되고, 격차가 작다면 서빙 비용을 감안해 선형모델을 재고할 근거가 된다.
- **실제 운영 반영 여부**: 현재 운영 대시보드(`mlreg.py`)는 GradientBoost 기준으로 설정돼 있다(§Part A-2) — 이 비교 결과, 더 나은 모델이 확인되면 운영 코드 교체 여부를 별도 논의가 필요한 사항으로 남긴다(이 노트북은 분석·비교까지만 다루고, 운영 코드 변경은 범위 밖이다).

## Part K. 신규 모델 내부테스트(자체 검증, TC-12 ~ TC-15)

Part D(TC-1~TC-7, GradientBoost 대상)는 수정하지 않고, 이번에 새로 추가한 4개 모델(XGBoost/LightGBM/LinearRegression/LogisticRegression)의 계산 결과만 별도로 검증한다. 검증 원칙은 Part D와 동일 — "특정 표본 크기나 실행 결과에 의존하지 않는, 값의 정합성·범위 검증"이다.

```mermaid
flowchart TD
    A["TC-12: 신규 회귀 3종(XGB/LGBM/LinReg)<br/>RMSLE≥0, R²≤1"] --> E{"전부 통과?"}
    B["TC-13: 신규 분류 3종(XGB/LGBM/LogReg)<br/>Accuracy·AUC·Recall·F1이 0~1"] --> E
    C["TC-14: 신규 모델 예측값에<br/>NaN/inf 없음"] --> E
    D["TC-15: LinearRegression 클리핑 후<br/>가격 예측값이 전부 0 이상"] --> E
    E -- "예" --> F["✅ Part K 내부테스트 통과"]
    E -- "아니오" --> G["❌ AssertionError로 즉시 중단"]

    style F fill:#E4F5E9,stroke:#2E8B57
    style G fill:#FFE0E0,stroke:#C0392B
```

In [ ]:
# 기술적 의미: 신규 회귀 모델 3종(XGBoost/LightGBM/LinearRegression)의 RMSLE가 정의상 0 이상인지, R²가 1을 초과하지 않는지 확인한다.
# 업무적 의미: Part D(TC-2/TC-3)와 동일한 원칙을 신규 모델 전체에 동일하게 적용해, "새 모델을 추가하면서 검증을 빠뜨리지 않는다"는 원칙을 유지한다.
assert rmsle_xgb >= 0, 'TC-12 실패: XGBoost RMSLE가 음수'
assert r2_xgb <= 1.0, 'TC-12 실패: XGBoost R2가 1을 초과함'
assert rmsle_lgbm >= 0, 'TC-12 실패: LightGBM RMSLE가 음수'
assert r2_lgbm <= 1.0, 'TC-12 실패: LightGBM R2가 1을 초과함'
assert rmsle_linreg >= 0, 'TC-12 실패: LinearRegression RMSLE가 음수'
assert r2_linreg <= 1.0, 'TC-12 실패: LinearRegression R2가 1을 초과함'

# 기술적 의미: 신규 분류 모델 3종(XGBoost/LightGBM/LogisticRegression)의 Accuracy·AUC·Recall(macro)·F1(macro)이 전부 정의상 0~1 범위인지 확인한다.
# 업무적 의미: Part D(TC-4/TC-7)와 동일한 원칙을 신규 모델 전체에 적용한다.
for name, a, u, rc, f1v in [
    ('XGBoost', acc_xgb, auc_xgb, recall_xgb, f1_xgb),
    ('LightGBM', acc_lgbm, auc_lgbm, recall_lgbm, f1_lgbm),
    ('LogisticRegression', acc_logreg, auc_logreg, recall_logreg, f1_logreg),
]:
    assert 0.0 <= a <= 1.0, f'TC-13 실패: {name} Accuracy가 0~1 범위를 벗어남'
    assert 0.0 <= u <= 1.0, f'TC-13 실패: {name} AUC가 0~1 범위를 벗어남'
    assert 0.0 <= rc <= 1.0, f'TC-13 실패: {name} Recall(macro)이 0~1 범위를 벗어남'
    assert 0.0 <= f1v <= 1.0, f'TC-13 실패: {name} F1-score(macro)가 0~1 범위를 벗어남'

# 기술적 의미: 신규 회귀 모델의 예측값(preds_xgb/preds_lgbm/preds_linreg)과 분류 모델의 확률 예측값(pred_proba_xgb/pred_proba_lgbm/pred_proba_logreg)에 NaN·inf가 없는지 확인한다.
# 업무적 의미: Part D(TC-6)와 동일한 "실제 서비스에 내보내기 전 마지막 품질 검사"를 신규 모델 전체에 적용한다.
assert np.isfinite(preds_xgb).all(), 'TC-14 실패: XGBoost 회귀 예측값에 NaN 또는 inf 존재'
assert np.isfinite(preds_lgbm).all(), 'TC-14 실패: LightGBM 회귀 예측값에 NaN 또는 inf 존재'
assert np.isfinite(preds_linreg).all(), 'TC-14 실패: LinearRegression 회귀 예측값에 NaN 또는 inf 존재'
assert np.isfinite(pred_proba_xgb).all(), 'TC-14 실패: XGBoost 분류 확률 예측값에 NaN 또는 inf 존재'
assert np.isfinite(pred_proba_lgbm).all(), 'TC-14 실패: LightGBM 분류 확률 예측값에 NaN 또는 inf 존재'
assert np.isfinite(pred_proba_logreg).all(), 'TC-14 실패: LogisticRegression 분류 확률 예측값에 NaN 또는 inf 존재'

# 기술적 의미: LinearRegression은 원리상 음수 가격을 예측할 수 있어 §Part H에서 0 이상으로 clip 처리했다 — 그 결과(preds_price_linreg)가 실제로 전부 0 이상인지 재확인한다.
# 업무적 의미: "음수 가격"이라는, 업무적으로 절대 있을 수 없는 값이 평가·보고서에 섞여 들어가지 않았음을 코드로 증명하는 단계다.
assert (preds_price_linreg >= 0).all(), 'TC-15 실패: LinearRegression 가격 예측값(clip 후)에 음수 존재'

# 기술적 의미: 위 4개 검증(TC-12~TC-15)이 모두 통과했다는 안내 문구를 출력한다.
print('TC-12 ~ TC-15 전부 통과 — Part K(신규 4개 모델) 내부테스트 성공')

# 기술적 의미: 신규 회귀 3종의 핵심 결과를 요약해 출력한다.
print(f'  - 회귀: XGB(RMSLE={rmsle_xgb:.4f}, R2={r2_xgb:.4f}) / LGBM(RMSLE={rmsle_lgbm:.4f}, R2={r2_lgbm:.4f}) / LinReg(RMSLE={rmsle_linreg:.4f}, R2={r2_linreg:.4f})')

# 기술적 의미: 신규 분류 3종의 핵심 결과를 요약해 출력한다.
print(f'  - 분류: XGB(ACC={acc_xgb:.4f}, AUC={auc_xgb:.4f}) / LGBM(ACC={acc_lgbm:.4f}, AUC={auc_lgbm:.4f}) / LogReg(ACC={acc_logreg:.4f}, AUC={auc_logreg:.4f})')


## 최종 결론 요약(5개 모델 종합, Part F~K 추가분)

- 기존 Part A~E(GradientBoost, 전체 데이터 1,482,535행 기준 실행 완료)는 손대지 않고, 이 지점부터 XGBoost(Part F)·LightGBM(Part G)·LinearRegression(Part H)·LogisticRegression(Part I)을 새로 추가했다 — 5개 모델 전부 **동일한 피처 행렬(`X_all`)과 동일한 학습/검증 분할**을 재사용해 공정하게 비교되도록 설계했다.
- 회귀·분류를 모두 지원하는 3개 모델(GradientBoost/XGBoost/LightGBM)은 `n_estimators=100, max_depth=3, learning_rate=0.05, random_state=156`으로 하이퍼파라미터를 통일했고, LightGBM은 리프 단위 성장 특성상 `num_leaves=8`을 추가로 맞췄다(§G-2).
- LinearRegression·LogisticRegression은 트리 계열과 근본적으로 다른 원리(선형결합)를 쓰는 모델이라 "맞출 하이퍼파라미터가 없다"는 점 자체를 §H-2/§I-2에서 명시했고, 대신 대용량 희소 데이터에 적합한 실무 표준 설정(예: `solver='liblinear'`)을 채택했다.
- Part J에서 5개 모델의 회귀 지표(RMSLE/R²/학습시간)와 분류 지표(Accuracy/AUC/Recall/F1/학습시간)를 각각 하나의 표로 모아 바로 비교할 수 있게 했다.
- Part K의 자체 검증(TC-12~TC-15)은 Part D(TC-1~TC-7)와 동일한 원칙(값의 정합성·범위 검증)을 신규 4개 모델에 확장 적용했다.
- **실행 상태**: Part F~K는 이번 세션에서 소스만 작성했고 아직 실행하지 않았다 — 이 노트북을 처음부터(Part 1) 순서대로 실행해야 Part F~K의 모든 `print`·`assert`가 실제 값으로 채워진다. GradientBoost(Part A~D)의 학습 소요시간(회귀 13,992.6초·분류 14,065.9초, 약 3.9시간)을 참고하면, XGBoost·LightGBM은 히스토그램 기반 구현체라 이보다 짧게 걸릴 가능성이 높고, LinearRegression·LogisticRegression은 트리 앙상블보다 훨씬 빠를 것으로 예상되지만, 정확한 소요시간은 실제 실행 후에만 확인할 수 있다.